In [1]:

# Milestone 2 - Imports


from pathlib import Path
import json
import joblib
import warnings

import numpy as np
import pandas as pd

from scipy import sparse
from scipy.sparse import load_npz

import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

print("Libraries imported successfully.")

Libraries imported successfully.


In [3]:
# =========================
# Project paths
# =========================

PROJECT_ROOT = Path.home() / "Documents" / "recommender_project" / "trial three"

PROCESSED_DIR = PROJECT_ROOT / "data_preprocessed"
MODELS_DIR = PROJECT_ROOT / "models"
REPORTS_DIR = PROJECT_ROOT / "reports"

MODELS_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("PROCESSED_DIR:", PROCESSED_DIR)
print("MODELS_DIR:", MODELS_DIR)
print("REPORTS_DIR:", REPORTS_DIR)

PROJECT_ROOT: C:\Users\pc\Documents\recommender_project\trial two
PROCESSED_DIR: C:\Users\pc\Documents\recommender_project\trial two\data_preprocessed
MODELS_DIR: C:\Users\pc\Documents\recommender_project\trial two\models
REPORTS_DIR: C:\Users\pc\Documents\recommender_project\trial two\reports


In [4]:
# =========================
# Load Milestone 1 artifacts
# =========================

# Interaction files
train_interactions = pd.read_parquet(
    PROCESSED_DIR / "train_interactions_encoded.parquet"
)

validation_interactions = pd.read_parquet(
    PROCESSED_DIR / "validation_interactions_known_encoded.parquet"
)

test_interactions = pd.read_parquet(
    PROCESSED_DIR / "test_interactions_known_encoded.parquet"
)

# Sparse train matrix
train_user_item_matrix = load_npz(
    PROCESSED_DIR / "train_user_item_matrix.npz"
)

# Encoders
user_encoder = joblib.load(PROCESSED_DIR / "user_encoder.pkl")
item_encoder = joblib.load(PROCESSED_DIR / "item_encoder.pkl")

# Seen items
user_seen_items_idx = joblib.load(
    PROCESSED_DIR / "user_seen_items_idx.pkl"
)

# Ground truth dictionaries
validation_gt_all = joblib.load(
    PROCESSED_DIR / "validation_gt_all_dict.pkl"
)

test_gt_all = joblib.load(
    PROCESSED_DIR / "test_gt_all_dict.pkl"
)

validation_gt_high_intent = joblib.load(
    PROCESSED_DIR / "validation_gt_high_intent_dict.pkl"
)

test_gt_high_intent = joblib.load(
    PROCESSED_DIR / "test_gt_high_intent_dict.pkl"
)

# Item metadata for content-based model later
item_features = pd.read_parquet(
    PROCESSED_DIR / "item_features.parquet"
)

item_tfidf_matrix = load_npz(
    PROCESSED_DIR / "item_tfidf_matrix.npz"
)

print("All Milestone 1 artifacts loaded successfully.")

All Milestone 1 artifacts loaded successfully.


In [5]:
# =========================
# Sanity checks
# =========================

print("===== DATA SHAPES =====")
print("Train interactions:", train_interactions.shape)
print("Validation interactions:", validation_interactions.shape)
print("Test interactions:", test_interactions.shape)

print("\nTrain user-item matrix:", train_user_item_matrix.shape)
print("Matrix non-zero values:", train_user_item_matrix.nnz)

print("\nNumber of users in encoder:", len(user_encoder.classes_))
print("Number of items in encoder:", len(item_encoder.classes_))

print("\nGround truth users:")
print("Validation all:", len(validation_gt_all))
print("Validation high-intent:", len(validation_gt_high_intent))
print("Test all:", len(test_gt_all))
print("Test high-intent:", len(test_gt_high_intent))

print("\nItem features:", item_features.shape)
print("Item TF-IDF matrix:", item_tfidf_matrix.shape)

print("\nExample train rows:")
display(train_interactions.head())

===== DATA SHAPES =====
Train interactions: (155049, 12)
Validation interactions: (46825, 12)
Test interactions: (46763, 12)

Train user-item matrix: (22890, 18224)
Matrix non-zero values: 155049

Number of users in encoder: 22890
Number of items in encoder: 18224

Ground truth users:
Validation all: 22869
Validation high-intent: 2260
Test all: 22860
Test high-intent: 3178

Item features: (18224, 5)
Item TF-IDF matrix: (18224, 300)

Example train rows:


,visitorid,itemid,interaction_strength,raw_event_strength,num_events,max_event_weight,first_interaction,last_interaction,is_high_intent,split,user_idx,item_idx
0,64,141657,0.113404,1.0,1,1.0,2015-06-15 21:51:51.942,2015-06-15 21:51:51.942,False,train,0,5543
1,64,233200,0.113491,1.0,1,1.0,2015-06-15 22:40:04.268,2015-06-15 22:40:04.268,False,train,0,9147
2,64,118401,0.113501,1.0,1,1.0,2015-06-15 22:45:31.466,2015-06-15 22:45:31.466,False,train,0,4640
3,155,123027,0.435783,1.0,1,1.0,2015-08-13 04:12:28.331,2015-08-13 04:12:28.331,False,train,1,4812
4,155,134620,0.871564,2.0,2,1.0,2015-08-13 04:11:43.687,2015-08-13 04:12:56.509,False,train,1,5261


In [6]:
# =========================
# Evaluation metrics for top-K recommendation
# =========================

def precision_at_k(recommended_items, relevant_items, k):
    """
    Precision@K = relevant recommended items / K
    """
    recommended_k = recommended_items[:k]
    relevant_set = set(relevant_items)

    if len(recommended_k) == 0:
        return 0.0

    hits = len(set(recommended_k) & relevant_set)
    return hits / k


def recall_at_k(recommended_items, relevant_items, k):
    """
    Recall@K = relevant recommended items / total relevant items
    """
    recommended_k = recommended_items[:k]
    relevant_set = set(relevant_items)

    if len(relevant_set) == 0:
        return 0.0

    hits = len(set(recommended_k) & relevant_set)
    return hits / len(relevant_set)


def average_precision_at_k(recommended_items, relevant_items, k):
    """
    AP@K measures whether relevant items appear early in the ranking.
    """
    recommended_k = recommended_items[:k]
    relevant_set = set(relevant_items)

    if len(relevant_set) == 0:
        return 0.0

    score = 0.0
    hits = 0

    for i, item in enumerate(recommended_k, start=1):
        if item in relevant_set:
            hits += 1
            score += hits / i

    return score / min(len(relevant_set), k)


def ndcg_at_k(recommended_items, relevant_items, k):
    """
    NDCG@K rewards putting relevant items near the top.
    """
    recommended_k = recommended_items[:k]
    relevant_set = set(relevant_items)

    dcg = 0.0

    for i, item in enumerate(recommended_k, start=1):
        if item in relevant_set:
            dcg += 1 / np.log2(i + 1)

    ideal_hits = min(len(relevant_set), k)

    if ideal_hits == 0:
        return 0.0

    idcg = sum(1 / np.log2(i + 1) for i in range(1, ideal_hits + 1))

    return dcg / idcg


def evaluate_recommendations(recommendations, ground_truth, num_items, k_values=[5, 10, 20]):
    """
    Evaluate recommendations using Precision@K, Recall@K, MAP@K, NDCG@K, and Coverage.
    """
    results = {}

    evaluated_users = list(ground_truth.keys())

    for k in k_values:
        precision_scores = []
        recall_scores = []
        map_scores = []
        ndcg_scores = []

        recommended_unique_items = set()

        for user_idx in evaluated_users:
            relevant_items = ground_truth[user_idx]
            recommended_items = recommendations.get(user_idx, [])

            recommended_unique_items.update(recommended_items[:k])

            precision_scores.append(
                precision_at_k(recommended_items, relevant_items, k)
            )

            recall_scores.append(
                recall_at_k(recommended_items, relevant_items, k)
            )

            map_scores.append(
                average_precision_at_k(recommended_items, relevant_items, k)
            )

            ndcg_scores.append(
                ndcg_at_k(recommended_items, relevant_items, k)
            )

        results[f"Precision@{k}"] = np.mean(precision_scores)
        results[f"Recall@{k}"] = np.mean(recall_scores)
        results[f"MAP@{k}"] = np.mean(map_scores)
        results[f"NDCG@{k}"] = np.mean(ndcg_scores)
        results[f"Coverage@{k}"] = len(recommended_unique_items) / num_items

    return results


print("Evaluation functions are ready.")

Evaluation functions are ready.


In [7]:
# =========================
# Popularity Baseline
# =========================

def build_popularity_ranking(train_df, score_col="interaction_strength"):
    """
    Rank items by total interaction strength in the training data.
    """
    popularity_scores = (
        train_df
        .groupby("item_idx")[score_col]
        .sum()
        .sort_values(ascending=False)
    )

    popular_items = popularity_scores.index.tolist()

    return popular_items, popularity_scores


def generate_popularity_recommendations(
    users,
    popular_items,
    user_seen_items_idx,
    k=20
):
    """
    Generate popularity recommendations while filtering already-seen items.
    """
    recommendations = {}

    for user_idx in users:
        seen_items = set(user_seen_items_idx.get(user_idx, []))

        user_recs = []

        for item_idx in popular_items:
            if item_idx not in seen_items:
                user_recs.append(item_idx)

            if len(user_recs) >= k:
                break

        recommendations[user_idx] = user_recs

    return recommendations


popular_items, popularity_scores = build_popularity_ranking(
    train_interactions,
    score_col="interaction_strength"
)

print("Popularity ranking created.")
print("Number of ranked items:", len(popular_items))

print("\nTop 10 popular item_idx:")
print(popular_items[:10])

print("\nTop 10 popularity scores:")
display(popularity_scores.head(10))

Popularity ranking created.
Number of ranked items: 18224

Top 10 popular item_idx:
[18020, 12255, 388, 8374, 17780, 26, 12529, 9478, 410, 9203]

Top 10 popularity scores:


item_idx
18020    200.734192
12255    156.494308
388      153.443085
8374     113.517433
17780    102.598999
26        98.628227
12529     94.533043
9478      93.922791
410       90.464149
9203      89.933266
Name: interaction_strength, dtype: float32

In [8]:
# =========================
# Evaluate Popularity Baseline on validation data
# =========================

NUM_ITEMS = train_user_item_matrix.shape[1]
K_MAX = 20

# Users for all-interaction validation
validation_users_all = list(validation_gt_all.keys())

popularity_recs_validation_all = generate_popularity_recommendations(
    users=validation_users_all,
    popular_items=popular_items,
    user_seen_items_idx=user_seen_items_idx,
    k=K_MAX
)

popularity_results_validation_all = evaluate_recommendations(
    recommendations=popularity_recs_validation_all,
    ground_truth=validation_gt_all,
    num_items=NUM_ITEMS,
    k_values=[5, 10, 20]
)

# Users for high-intent validation
validation_users_high_intent = list(validation_gt_high_intent.keys())

popularity_recs_validation_high_intent = generate_popularity_recommendations(
    users=validation_users_high_intent,
    popular_items=popular_items,
    user_seen_items_idx=user_seen_items_idx,
    k=K_MAX
)

popularity_results_validation_high_intent = evaluate_recommendations(
    recommendations=popularity_recs_validation_high_intent,
    ground_truth=validation_gt_high_intent,
    num_items=NUM_ITEMS,
    k_values=[5, 10, 20]
)

print("===== POPULARITY BASELINE - VALIDATION ALL =====")
display(pd.DataFrame([popularity_results_validation_all]))

print("\n===== POPULARITY BASELINE - VALIDATION HIGH INTENT =====")
display(pd.DataFrame([popularity_results_validation_high_intent]))

===== POPULARITY BASELINE - VALIDATION ALL =====


,Precision@5,Recall@5,MAP@5,NDCG@5,Coverage@5,Precision@10,Recall@10,MAP@10,NDCG@10,Coverage@10,Precision@20,Recall@20,MAP@20,NDCG@20,Coverage@20
0,0.001548,0.003818,0.001951,0.002893,0.000604,0.001526,0.007657,0.002384,0.0042,0.001317,0.001279,0.012126,0.002664,0.005448,0.00214



===== POPULARITY BASELINE - VALIDATION HIGH INTENT =====


,Precision@5,Recall@5,MAP@5,NDCG@5,Coverage@5,Precision@10,Recall@10,MAP@10,NDCG@10,Coverage@10,Precision@20,Recall@20,MAP@20,NDCG@20,Coverage@20
0,0.004779,0.01672,0.010685,0.013076,0.000604,0.003584,0.023781,0.011498,0.015406,0.001317,0.002721,0.033703,0.012149,0.018037,0.00214


In [9]:
# =========================
# Print Popularity results clearly
# =========================

def print_results(title, results):
    """
    Print evaluation results in a clean readable format.
    """
    print("\n" + "=" * 60)
    print(title)
    print("=" * 60)
    
    for metric, value in results.items():
        print(f"{metric}: {value:.6f}")


print("validation_gt_all users:", len(validation_gt_all))
print("validation_gt_high_intent users:", len(validation_gt_high_intent))

print_results(
    "POPULARITY - VALIDATION ALL",
    popularity_results_validation_all
)

print_results(
    "POPULARITY - VALIDATION HIGH INTENT",
    popularity_results_validation_high_intent
)

validation_gt_all users: 22869
validation_gt_high_intent users: 2260

POPULARITY - VALIDATION ALL
Precision@5: 0.001548
Recall@5: 0.003818
MAP@5: 0.001951
NDCG@5: 0.002893
Coverage@5: 0.000604
Precision@10: 0.001526
Recall@10: 0.007657
MAP@10: 0.002384
NDCG@10: 0.004200
Coverage@10: 0.001317
Precision@20: 0.001279
Recall@20: 0.012126
MAP@20: 0.002664
NDCG@20: 0.005448
Coverage@20: 0.002140

POPULARITY - VALIDATION HIGH INTENT
Precision@5: 0.004779
Recall@5: 0.016720
MAP@5: 0.010685
NDCG@5: 0.013076
Coverage@5: 0.000604
Precision@10: 0.003584
Recall@10: 0.023781
MAP@10: 0.011498
NDCG@10: 0.015406
Coverage@10: 0.001317
Precision@20: 0.002721
Recall@20: 0.033703
MAP@20: 0.012149
NDCG@20: 0.018037
Coverage@20: 0.002140


In [10]:
# =========================
# Save Popularity Baseline results
# =========================

popularity_validation_all_row = {
    "model": "Popularity",
    "evaluation_type": "validation_all",
    **popularity_results_validation_all
}

popularity_validation_high_intent_row = {
    "model": "Popularity",
    "evaluation_type": "validation_high_intent",
    **popularity_results_validation_high_intent
}

popularity_results_df = pd.DataFrame([
    popularity_validation_all_row,
    popularity_validation_high_intent_row
])

popularity_results_path = REPORTS_DIR / "popularity_baseline_validation_results.csv"

popularity_results_df.to_csv(popularity_results_path, index=False)

print("Popularity results saved to:")
print(popularity_results_path)

display(popularity_results_df)

Popularity results saved to:
C:\Users\pc\Documents\recommender_project\trial two\reports\popularity_baseline_validation_results.csv


,model,evaluation_type,Precision@5,Recall@5,MAP@5,NDCG@5,Coverage@5,Precision@10,Recall@10,MAP@10,NDCG@10,Coverage@10,Precision@20,Recall@20,MAP@20,NDCG@20,Coverage@20
0,Popularity,validation_all,0.001548,0.003818,0.001951,0.002893,0.000604,0.001526,0.007657,0.002384,0.004200,0.001317,0.001279,0.012126,0.002664,0.005448,0.00214
1,Popularity,validation_high_intent,0.004779,0.016720,0.010685,0.013076,0.000604,0.003584,0.023781,0.011498,0.015406,0.001317,0.002721,0.033703,0.012149,0.018037,0.00214


In [11]:
# =========================
# ALS - Install and import
# =========================

# Run this only if implicit is not installed
# !pip install implicit

from implicit.als import AlternatingLeastSquares

print("implicit ALS imported successfully.")

implicit ALS imported successfully.


In [12]:
# =========================
# ALS - Train initial model
# =========================

ALS_FACTORS = 64
ALS_REGULARIZATION = 0.05
ALS_ITERATIONS = 20
ALS_ALPHA = 40

# implicit ALS uses confidence-weighted interactions
als_train_matrix = train_user_item_matrix * ALS_ALPHA

als_model = AlternatingLeastSquares(
    factors=ALS_FACTORS,
    regularization=ALS_REGULARIZATION,
    iterations=ALS_ITERATIONS,
    random_state=42
)

als_model.fit(als_train_matrix)

print("ALS model trained successfully.")
print("Factors:", ALS_FACTORS)
print("Regularization:", ALS_REGULARIZATION)
print("Iterations:", ALS_ITERATIONS)
print("Alpha:", ALS_ALPHA)

  0%|          | 0/20 [00:00<?, ?it/s]

ALS model trained successfully.
Factors: 64
Regularization: 0.05
Iterations: 20
Alpha: 40


In [13]:
# =========================
# ALS - Generate recommendations
# =========================

def generate_als_recommendations(model, user_ids, user_item_matrix, k=20):
    """
    Generate ALS recommendations for a list of encoded user_idx values.
    """
    recommendations = {}

    for user_idx in user_ids:
        recommended_items, scores = model.recommend(
            userid=user_idx,
            user_items=user_item_matrix[user_idx],
            N=k,
            filter_already_liked_items=True
        )

        recommendations[user_idx] = recommended_items.tolist()

    return recommendations


print("ALS recommendation function is ready.")

ALS recommendation function is ready.


In [14]:
# =========================
# ALS - Evaluate on validation
# =========================

NUM_ITEMS = train_user_item_matrix.shape[1]
K_MAX = 20

# Validation ALL
validation_users_all = list(validation_gt_all.keys())

als_recs_validation_all = generate_als_recommendations(
    model=als_model,
    user_ids=validation_users_all,
    user_item_matrix=train_user_item_matrix,
    k=K_MAX
)

als_results_validation_all = evaluate_recommendations(
    recommendations=als_recs_validation_all,
    ground_truth=validation_gt_all,
    num_items=NUM_ITEMS,
    k_values=[5, 10, 20]
)

# Validation HIGH INTENT
validation_users_high_intent = list(validation_gt_high_intent.keys())

als_recs_validation_high_intent = generate_als_recommendations(
    model=als_model,
    user_ids=validation_users_high_intent,
    user_item_matrix=train_user_item_matrix,
    k=K_MAX
)

als_results_validation_high_intent = evaluate_recommendations(
    recommendations=als_recs_validation_high_intent,
    ground_truth=validation_gt_high_intent,
    num_items=NUM_ITEMS,
    k_values=[5, 10, 20]
)

print_results(
    "ALS - VALIDATION ALL",
    als_results_validation_all
)

print_results(
    "ALS - VALIDATION HIGH INTENT",
    als_results_validation_high_intent
)


ALS - VALIDATION ALL
Precision@5: 0.026927
Recall@5: 0.087042
MAP@5: 0.047822
NDCG@5: 0.063113
Coverage@5: 0.169666
Precision@10: 0.020858
Recall@10: 0.132568
MAP@10: 0.054333
NDCG@10: 0.079436
Coverage@10: 0.228545
Precision@20: 0.015711
Recall@20: 0.196173
MAP@20: 0.059317
NDCG@20: 0.097464
Coverage@20: 0.305806

ALS - VALIDATION HIGH INTENT
Precision@5: 0.026549
Recall@5: 0.102812
MAP@5: 0.061604
NDCG@5: 0.075026
Coverage@5: 0.100417
Precision@10: 0.018938
Recall@10: 0.144381
MAP@10: 0.067386
NDCG@10: 0.089288
Coverage@10: 0.149967
Precision@20: 0.013473
Recall@20: 0.202244
MAP@20: 0.071717
NDCG@20: 0.104937
Coverage@20: 0.220314


In [15]:
# =========================
# Save ALS initial validation results
# =========================

als_validation_all_row = {
    "model": "ALS",
    "evaluation_type": "validation_all",
    "factors": ALS_FACTORS,
    "regularization": ALS_REGULARIZATION,
    "iterations": ALS_ITERATIONS,
    "alpha": ALS_ALPHA,
    **als_results_validation_all
}

als_validation_high_intent_row = {
    "model": "ALS",
    "evaluation_type": "validation_high_intent",
    "factors": ALS_FACTORS,
    "regularization": ALS_REGULARIZATION,
    "iterations": ALS_ITERATIONS,
    "alpha": ALS_ALPHA,
    **als_results_validation_high_intent
}

als_results_df = pd.DataFrame([
    als_validation_all_row,
    als_validation_high_intent_row
])

als_results_path = REPORTS_DIR / "als_initial_validation_results.csv"
als_results_df.to_csv(als_results_path, index=False)

print("ALS initial results saved to:")
print(als_results_path)

display(als_results_df)

ALS initial results saved to:
C:\Users\pc\Documents\recommender_project\trial two\reports\als_initial_validation_results.csv


,model,evaluation_type,factors,regularization,iterations,alpha,Precision@5,Recall@5,MAP@5,NDCG@5,...,Precision@10,Recall@10,MAP@10,NDCG@10,Coverage@10,Precision@20,Recall@20,MAP@20,NDCG@20,Coverage@20
0,ALS,validation_all,64,0.05,20,40,0.026927,0.087042,0.047822,0.063113,...,0.020858,0.132568,0.054333,0.079436,0.228545,0.015711,0.196173,0.059317,0.097464,0.305806
1,ALS,validation_high_intent,64,0.05,20,40,0.026549,0.102812,0.061604,0.075026,...,0.018938,0.144381,0.067386,0.089288,0.149967,0.013473,0.202244,0.071717,0.104937,0.220314


In [16]:
# =========================
# ALS - Small hyperparameter tuning
# =========================

als_tuning_configs = [
    {"factors": 32, "regularization": 0.05, "alpha": 20, "iterations": 20},
    {"factors": 64, "regularization": 0.05, "alpha": 40, "iterations": 20},
    {"factors": 128, "regularization": 0.05, "alpha": 40, "iterations": 20},
    {"factors": 64, "regularization": 0.01, "alpha": 40, "iterations": 20},
    {"factors": 64, "regularization": 0.10, "alpha": 40, "iterations": 20},
    {"factors": 64, "regularization": 0.05, "alpha": 80, "iterations": 20},
]

print("Number of ALS configs:", len(als_tuning_configs))
als_tuning_configs

Number of ALS configs: 6


[{'factors': 32, 'regularization': 0.05, 'alpha': 20, 'iterations': 20},
 {'factors': 64, 'regularization': 0.05, 'alpha': 40, 'iterations': 20},
 {'factors': 128, 'regularization': 0.05, 'alpha': 40, 'iterations': 20},
 {'factors': 64, 'regularization': 0.01, 'alpha': 40, 'iterations': 20},
 {'factors': 64, 'regularization': 0.1, 'alpha': 40, 'iterations': 20},
 {'factors': 64, 'regularization': 0.05, 'alpha': 80, 'iterations': 20}]

In [17]:
# =========================
# ALS - Tune on validation high-intent
# =========================

als_tuning_results = []

validation_users_hi = list(validation_gt_high_intent.keys())
NUM_ITEMS = train_user_item_matrix.shape[1]
K_MAX = 20

for i, config in enumerate(als_tuning_configs, start=1):
    print("\n" + "=" * 70)
    print(f"Training ALS config {i}/{len(als_tuning_configs)}")
    print(config)
    print("=" * 70)
    
    confidence_matrix = train_user_item_matrix * config["alpha"]
    
    model = AlternatingLeastSquares(
        factors=config["factors"],
        regularization=config["regularization"],
        iterations=config["iterations"],
        random_state=42
    )
    
    model.fit(confidence_matrix)
    
    recs_hi = generate_als_recommendations(
        model=model,
        user_ids=validation_users_hi,
        user_item_matrix=train_user_item_matrix,
        k=K_MAX
    )
    
    results_hi = evaluate_recommendations(
        recommendations=recs_hi,
        ground_truth=validation_gt_high_intent,
        num_items=NUM_ITEMS,
        k_values=[5, 10, 20]
    )
    
    row = {
        "model": "ALS",
        **config,
        **results_hi
    }
    
    als_tuning_results.append(row)
    
    print("NDCG@10:", round(results_hi["NDCG@10"], 6))
    print("Recall@10:", round(results_hi["Recall@10"], 6))
    print("Precision@10:", round(results_hi["Precision@10"], 6))
    print("Coverage@10:", round(results_hi["Coverage@10"], 6))


als_tuning_results_df = pd.DataFrame(als_tuning_results)

als_tuning_results_df = als_tuning_results_df.sort_values(
    by="NDCG@10",
    ascending=False
).reset_index(drop=True)

display(als_tuning_results_df)

als_tuning_results_path = REPORTS_DIR / "als_tuning_validation_high_intent.csv"
als_tuning_results_df.to_csv(als_tuning_results_path, index=False)

print("Saved ALS tuning results to:")
print(als_tuning_results_path)


Training ALS config 1/6
{'factors': 32, 'regularization': 0.05, 'alpha': 20, 'iterations': 20}


  0%|          | 0/20 [00:00<?, ?it/s]

NDCG@10: 0.064311
Recall@10: 0.09949
Precision@10: 0.013584
Coverage@10: 0.081431

Training ALS config 2/6
{'factors': 64, 'regularization': 0.05, 'alpha': 40, 'iterations': 20}


  0%|          | 0/20 [00:00<?, ?it/s]

NDCG@10: 0.089288
Recall@10: 0.144381
Precision@10: 0.018938
Coverage@10: 0.149967

Training ALS config 3/6
{'factors': 128, 'regularization': 0.05, 'alpha': 40, 'iterations': 20}


...

NDCG@10: 0.089524
Recall@10: 0.145993
Precision@10: 0.019159
Coverage@10: 0.150626

Training ALS config 6/6
{'factors': 64, 'regularization': 0.05, 'alpha': 80, 'iterations': 20}


  0%|          | 0/20 [00:00<?, ?it/s]

NDCG@10: 0.096096
Recall@10: 0.162644
Precision@10: 0.021327
Coverage@10: 0.168624


,model,factors,regularization,alpha,iterations,Precision@5,Recall@5,MAP@5,NDCG@5,Coverage@5,Precision@10,Recall@10,MAP@10,NDCG@10,Coverage@10,Precision@20,Recall@20,MAP@20,NDCG@20,Coverage@20
0,ALS,128,0.05,40,20,0.030619,0.118042,0.069371,0.085183,0.147278,0.022522,0.172582,0.076993,0.103734,0.224210,0.015819,0.239346,0.082016,0.121742,0.328139
1,ALS,64,0.05,80,20,0.029381,0.113446,0.063983,0.079633,0.110349,0.021327,0.162644,0.070433,0.096096,0.168624,0.014867,0.223619,0.074955,0.112508,0.254390
2,ALS,64,0.10,40,20,0.026726,0.103379,0.061318,0.074983,0.099704,0.019159,0.145993,0.067218,0.089524,0.150626,0.013230,0.199154,0.071161,0.103869,0.220588
3,ALS,64,0.05,40,20,0.026549,0.102812,0.061604,0.075026,0.100417,0.018938,0.144381,0.067386,0.089288,0.149967,0.013473,0.202244,0.071717,0.104937,0.220314
4,ALS,64,0.01,40,20,0.026195,0.102295,0.061334,0.074613,0.100088,0.018805,0.143447,0.067155,0.088889,0.149199,0.013341,0.201320,0.071566,0.104603,0.220588
5,ALS,32,0.05,20,20,0.018584,0.069698,0.045589,0.054099,0.055476,0.013584,0.099490,0.049688,0.064311,0.081431,0.009314,0.135774,0.052235,0.074011,0.119787


Saved ALS tuning results to:
C:\Users\pc\Documents\recommender_project\trial two\reports\als_tuning_validation_high_intent.csv


In [18]:
# =========================
# ALS - Focused tuning around best config
# =========================

als_tuning_configs_round_2 = [
    {"factors": 128, "regularization": 0.05, "alpha": 20, "iterations": 20},
    {"factors": 128, "regularization": 0.05, "alpha": 40, "iterations": 20},  # current best
    {"factors": 128, "regularization": 0.05, "alpha": 80, "iterations": 20},
    {"factors": 128, "regularization": 0.01, "alpha": 40, "iterations": 20},
    {"factors": 128, "regularization": 0.10, "alpha": 40, "iterations": 20},
    {"factors": 128, "regularization": 0.05, "alpha": 40, "iterations": 30},
]

als_tuning_results_round_2 = []

validation_users_hi = list(validation_gt_high_intent.keys())
NUM_ITEMS = train_user_item_matrix.shape[1]
K_MAX = 20

for i, config in enumerate(als_tuning_configs_round_2, start=1):
    print("\n" + "=" * 70)
    print(f"Training ALS Round 2 config {i}/{len(als_tuning_configs_round_2)}")
    print(config)
    print("=" * 70)

    # Apply confidence scaling
    confidence_matrix = train_user_item_matrix * config["alpha"]

    model = AlternatingLeastSquares(
        factors=config["factors"],
        regularization=config["regularization"],
        iterations=config["iterations"],
        random_state=42
    )

    model.fit(confidence_matrix)

    recs_hi = generate_als_recommendations(
        model=model,
        user_ids=validation_users_hi,
        user_item_matrix=train_user_item_matrix,
        k=K_MAX
    )

    results_hi = evaluate_recommendations(
        recommendations=recs_hi,
        ground_truth=validation_gt_high_intent,
        num_items=NUM_ITEMS,
        k_values=[5, 10, 20]
    )

    row = {
        "model": "ALS",
        **config,
        **results_hi
    }

    als_tuning_results_round_2.append(row)

    print("Precision@10:", round(results_hi["Precision@10"], 6))
    print("Recall@10:", round(results_hi["Recall@10"], 6))
    print("MAP@10:", round(results_hi["MAP@10"], 6))
    print("NDCG@10:", round(results_hi["NDCG@10"], 6))
    print("Coverage@10:", round(results_hi["Coverage@10"], 6))


als_tuning_results_round_2_df = pd.DataFrame(als_tuning_results_round_2)

als_tuning_results_round_2_df = als_tuning_results_round_2_df.sort_values(
    by="NDCG@10",
    ascending=False
).reset_index(drop=True)

display(als_tuning_results_round_2_df)

als_round_2_path = REPORTS_DIR / "als_tuning_round_2_validation_high_intent.csv"
als_tuning_results_round_2_df.to_csv(als_round_2_path, index=False)

print("Saved ALS Round 2 tuning results to:")
print(als_round_2_path)


Training ALS Round 2 config 1/6
{'factors': 128, 'regularization': 0.05, 'alpha': 20, 'iterations': 20}


  0%|          | 0/20 [00:00<?, ?it/s]

Precision@10: 0.020177
Recall@10: 0.154897
MAP@10: 0.071698
NDCG@10: 0.09522
Coverage@10: 0.200285

Training ALS Round 2 config 2/6
{'factors': 128, 'regularization': 0.05, 'alpha': 40, 'iterations': 20}


  0%|          | 0/20 [00:00<?, ?it/s]

Precision@10: 0.022522
Recall@10: 0.172582
MAP@10: 0.076993
NDCG@10: 0.103734
Coverage@10: 0.22421

Training ALS Round 2 config 3/6
{'factors': 128, 'regularization': 0.05, 'alpha': 80, 'iterations': 20}


...

Precision@10: 0.022611
Recall@10: 0.172947
MAP@10: 0.076476
NDCG@10: 0.103443
Coverage@10: 0.223661

Training ALS Round 2 config 6/6
{'factors': 128, 'regularization': 0.05, 'alpha': 40, 'iterations': 30}


  0%|          | 0/30 [00:00<?, ?it/s]

Precision@10: 0.022434
Recall@10: 0.17352
MAP@10: 0.076125
NDCG@10: 0.103172
Coverage@10: 0.219491


,model,factors,regularization,alpha,iterations,Precision@5,Recall@5,MAP@5,NDCG@5,Coverage@5,Precision@10,Recall@10,MAP@10,NDCG@10,Coverage@10,Precision@20,Recall@20,MAP@20,NDCG@20,Coverage@20
0,ALS,128,0.05,80,20,0.033540,0.131038,0.072501,0.090984,0.160119,0.023850,0.184149,0.079939,0.108998,0.246049,0.016571,0.252303,0.084962,0.127279,0.363641
1,ALS,128,0.01,40,20,0.030708,0.119345,0.069894,0.085835,0.147004,0.022699,0.173911,0.077417,0.104358,0.224320,0.015885,0.240018,0.082427,0.122233,0.327371
2,ALS,128,0.05,40,20,0.030619,0.118042,0.069371,0.085183,0.147278,0.022522,0.172582,0.076993,0.103734,0.224210,0.015819,0.239346,0.082016,0.121742,0.328139
3,ALS,128,0.10,40,20,0.030531,0.117821,0.068725,0.084605,0.146620,0.022611,0.172947,0.076476,0.103443,0.223661,0.015863,0.240191,0.081481,0.121522,0.327371
4,ALS,128,0.05,40,30,0.030177,0.117891,0.068467,0.084337,0.144754,0.022434,0.173520,0.076125,0.103172,0.219491,0.015796,0.240848,0.081112,0.121248,0.322322
5,ALS,128,0.05,20,20,0.027699,0.108538,0.065253,0.079296,0.134932,0.020177,0.154897,0.071698,0.095220,0.200285,0.014248,0.214885,0.076098,0.111384,0.289947


Saved ALS Round 2 tuning results to:
C:\Users\pc\Documents\recommender_project\trial two\reports\als_tuning_round_2_validation_high_intent.csv


In [19]:
# =========================
# ALS - Final focused tuning
# =========================

als_final_configs = [
    {"factors": 128, "regularization": 0.05, "alpha": 100, "iterations": 20},
    {"factors": 128, "regularization": 0.03, "alpha": 80, "iterations": 20},
    {"factors": 160, "regularization": 0.05, "alpha": 80, "iterations": 20},
    {"factors": 192, "regularization": 0.05, "alpha": 80, "iterations": 20},
]

als_final_results = []

validation_users_hi = list(validation_gt_high_intent.keys())
NUM_ITEMS = train_user_item_matrix.shape[1]
K_MAX = 20

for i, config in enumerate(als_final_configs, start=1):
    print("\n" + "=" * 70)
    print(f"Training ALS final config {i}/{len(als_final_configs)}")
    print(config)
    print("=" * 70)

    # Apply confidence scaling
    confidence_matrix = train_user_item_matrix * config["alpha"]

    model = AlternatingLeastSquares(
        factors=config["factors"],
        regularization=config["regularization"],
        iterations=config["iterations"],
        random_state=42
    )

    model.fit(confidence_matrix)

    recs_hi = generate_als_recommendations(
        model=model,
        user_ids=validation_users_hi,
        user_item_matrix=train_user_item_matrix,
        k=K_MAX
    )

    results_hi = evaluate_recommendations(
        recommendations=recs_hi,
        ground_truth=validation_gt_high_intent,
        num_items=NUM_ITEMS,
        k_values=[5, 10, 20]
    )

    row = {
        "model": "ALS",
        **config,
        **results_hi
    }

    als_final_results.append(row)

    print("Precision@10:", round(results_hi["Precision@10"], 6))
    print("Recall@10:", round(results_hi["Recall@10"], 6))
    print("MAP@10:", round(results_hi["MAP@10"], 6))
    print("NDCG@10:", round(results_hi["NDCG@10"], 6))
    print("Coverage@10:", round(results_hi["Coverage@10"], 6))


als_final_results_df = pd.DataFrame(als_final_results)

als_final_results_df = als_final_results_df.sort_values(
    by="NDCG@10",
    ascending=False
).reset_index(drop=True)

display(als_final_results_df)

als_final_path = REPORTS_DIR / "als_final_tuning_validation_high_intent.csv"
als_final_results_df.to_csv(als_final_path, index=False)

print("Saved ALS final tuning results to:")
print(als_final_path)


Training ALS final config 1/4
{'factors': 128, 'regularization': 0.05, 'alpha': 100, 'iterations': 20}


  0%|          | 0/20 [00:00<?, ?it/s]

Precision@10: 0.024381
Recall@10: 0.18802
MAP@10: 0.080927
NDCG@10: 0.110575
Coverage@10: 0.254719

Training ALS final config 2/4
{'factors': 128, 'regularization': 0.03, 'alpha': 80, 'iterations': 20}


  0%|          | 0/20 [00:00<?, ?it/s]

Precision@10: 0.024159
Recall@10: 0.186185
MAP@10: 0.080874
NDCG@10: 0.110185
Coverage@10: 0.246049

Training ALS final config 3/4
{'factors': 160, 'regularization': 0.05, 'alpha': 80, 'iterations': 20}


...

Precision@10: 0.024469
Recall@10: 0.190288
MAP@10: 0.081232
NDCG@10: 0.111269
Coverage@10: 0.274144

Training ALS final config 4/4
{'factors': 192, 'regularization': 0.05, 'alpha': 80, 'iterations': 20}


  0%|          | 0/20 [00:00<?, ?it/s]

Precision@10: 0.025265
Recall@10: 0.196525
MAP@10: 0.084815
NDCG@10: 0.115658
Coverage@10: 0.298178


,model,factors,regularization,alpha,iterations,Precision@5,Recall@5,MAP@5,NDCG@5,Coverage@5,Precision@10,Recall@10,MAP@10,NDCG@10,Coverage@10,Precision@20,Recall@20,MAP@20,NDCG@20,Coverage@20
0,ALS,192,0.05,80,20,0.034248,0.135462,0.076266,0.094882,0.194853,0.025265,0.196525,0.084815,0.115658,0.298178,0.018031,0.272887,0.090447,0.136211,0.433933
1,ALS,160,0.05,80,20,0.033009,0.130898,0.072646,0.090784,0.178720,0.024469,0.190288,0.081232,0.111269,0.274144,0.017434,0.264930,0.086852,0.131462,0.401613
2,ALS,128,0.05,100,20,0.033628,0.132591,0.073234,0.091742,0.165222,0.024381,0.188020,0.080927,0.110575,0.254719,0.016947,0.258487,0.086099,0.129404,0.375713
3,ALS,128,0.03,80,20,0.033717,0.131917,0.073363,0.091838,0.160503,0.024159,0.186185,0.080874,0.110185,0.246049,0.016615,0.252973,0.085746,0.128013,0.363202


Saved ALS final tuning results to:
C:\Users\pc\Documents\recommender_project\trial two\reports\als_final_tuning_validation_high_intent.csv


In [20]:
# =========================
# Show ALS final tuning results
# =========================

display(als_final_results_df)

print("\nBest ALS config:")
display(als_final_results_df.head(1))

,model,factors,regularization,alpha,iterations,Precision@5,Recall@5,MAP@5,NDCG@5,Coverage@5,Precision@10,Recall@10,MAP@10,NDCG@10,Coverage@10,Precision@20,Recall@20,MAP@20,NDCG@20,Coverage@20
0,ALS,192,0.05,80,20,0.034248,0.135462,0.076266,0.094882,0.194853,0.025265,0.196525,0.084815,0.115658,0.298178,0.018031,0.272887,0.090447,0.136211,0.433933
1,ALS,160,0.05,80,20,0.033009,0.130898,0.072646,0.090784,0.178720,0.024469,0.190288,0.081232,0.111269,0.274144,0.017434,0.264930,0.086852,0.131462,0.401613
2,ALS,128,0.05,100,20,0.033628,0.132591,0.073234,0.091742,0.165222,0.024381,0.188020,0.080927,0.110575,0.254719,0.016947,0.258487,0.086099,0.129404,0.375713
3,ALS,128,0.03,80,20,0.033717,0.131917,0.073363,0.091838,0.160503,0.024159,0.186185,0.080874,0.110185,0.246049,0.016615,0.252973,0.085746,0.128013,0.363202



Best ALS config:


,model,factors,regularization,alpha,iterations,Precision@5,Recall@5,MAP@5,NDCG@5,Coverage@5,Precision@10,Recall@10,MAP@10,NDCG@10,Coverage@10,Precision@20,Recall@20,MAP@20,NDCG@20,Coverage@20
0,ALS,192,0.05,80,20,0.034248,0.135462,0.076266,0.094882,0.194853,0.025265,0.196525,0.084815,0.115658,0.298178,0.018031,0.272887,0.090447,0.136211,0.433933


In [21]:
# =========================
# Train Best ALS Model
# =========================

BEST_ALS_FACTORS = 192
BEST_ALS_REGULARIZATION = 0.05
BEST_ALS_ALPHA = 80
BEST_ALS_ITERATIONS = 20

best_als_confidence_matrix = train_user_item_matrix * BEST_ALS_ALPHA

best_als_model = AlternatingLeastSquares(
    factors=BEST_ALS_FACTORS,
    regularization=BEST_ALS_REGULARIZATION,
    iterations=BEST_ALS_ITERATIONS,
    random_state=42
)

best_als_model.fit(best_als_confidence_matrix)

print("Best ALS model trained successfully.")
print("factors:", BEST_ALS_FACTORS)
print("regularization:", BEST_ALS_REGULARIZATION)
print("alpha:", BEST_ALS_ALPHA)
print("iterations:", BEST_ALS_ITERATIONS)

  0%|          | 0/20 [00:00<?, ?it/s]

Best ALS model trained successfully.
factors: 192
regularization: 0.05
alpha: 80
iterations: 20


In [22]:
# =========================
# Evaluate Best ALS on validation all + high-intent
# =========================

NUM_ITEMS = train_user_item_matrix.shape[1]
K_MAX = 20

# Validation ALL
validation_users_all = list(validation_gt_all.keys())

best_als_recs_validation_all = generate_als_recommendations(
    model=best_als_model,
    user_ids=validation_users_all,
    user_item_matrix=train_user_item_matrix,
    k=K_MAX
)

best_als_results_validation_all = evaluate_recommendations(
    recommendations=best_als_recs_validation_all,
    ground_truth=validation_gt_all,
    num_items=NUM_ITEMS,
    k_values=[5, 10, 20]
)

# Validation HIGH INTENT
validation_users_high_intent = list(validation_gt_high_intent.keys())

best_als_recs_validation_high_intent = generate_als_recommendations(
    model=best_als_model,
    user_ids=validation_users_high_intent,
    user_item_matrix=train_user_item_matrix,
    k=K_MAX
)

best_als_results_validation_high_intent = evaluate_recommendations(
    recommendations=best_als_recs_validation_high_intent,
    ground_truth=validation_gt_high_intent,
    num_items=NUM_ITEMS,
    k_values=[5, 10, 20]
)

print_results(
    "BEST ALS - VALIDATION ALL",
    best_als_results_validation_all
)

print_results(
    "BEST ALS - VALIDATION HIGH INTENT",
    best_als_results_validation_high_intent
)


BEST ALS - VALIDATION ALL
Precision@5: 0.039731
Recall@5: 0.132181
MAP@5: 0.073201
NDCG@5: 0.095338
Coverage@5: 0.373738
Precision@10: 0.030976
Recall@10: 0.202025
MAP@10: 0.083306
NDCG@10: 0.120246
Coverage@10: 0.487434
Precision@20: 0.023173
Recall@20: 0.297631
MAP@20: 0.090996
NDCG@20: 0.147223
Coverage@20: 0.610075

BEST ALS - VALIDATION HIGH INTENT
Precision@5: 0.034248
Recall@5: 0.135462
MAP@5: 0.076266
NDCG@5: 0.094882
Coverage@5: 0.194853
Precision@10: 0.025265
Recall@10: 0.196525
MAP@10: 0.084815
NDCG@10: 0.115658
Coverage@10: 0.298178
Precision@20: 0.018031
Recall@20: 0.272887
MAP@20: 0.090447
NDCG@20: 0.136211
Coverage@20: 0.433933


In [23]:
# =========================
# Evaluate Best ALS on validation all + high-intent
# =========================

NUM_ITEMS = train_user_item_matrix.shape[1]
K_MAX = 20

# Validation ALL
validation_users_all = list(validation_gt_all.keys())

best_als_recs_validation_all = generate_als_recommendations(
    model=best_als_model,
    user_ids=validation_users_all,
    user_item_matrix=train_user_item_matrix,
    k=K_MAX
)

best_als_results_validation_all = evaluate_recommendations(
    recommendations=best_als_recs_validation_all,
    ground_truth=validation_gt_all,
    num_items=NUM_ITEMS,
    k_values=[5, 10, 20]
)

# Validation HIGH INTENT
validation_users_high_intent = list(validation_gt_high_intent.keys())

best_als_recs_validation_high_intent = generate_als_recommendations(
    model=best_als_model,
    user_ids=validation_users_high_intent,
    user_item_matrix=train_user_item_matrix,
    k=K_MAX
)

best_als_results_validation_high_intent = evaluate_recommendations(
    recommendations=best_als_recs_validation_high_intent,
    ground_truth=validation_gt_high_intent,
    num_items=NUM_ITEMS,
    k_values=[5, 10, 20]
)

print_results(
    "BEST ALS - VALIDATION ALL",
    best_als_results_validation_all
)

print_results(
    "BEST ALS - VALIDATION HIGH INTENT",
    best_als_results_validation_high_intent
)


BEST ALS - VALIDATION ALL
Precision@5: 0.039731
Recall@5: 0.132181
MAP@5: 0.073201
NDCG@5: 0.095338
Coverage@5: 0.373738
Precision@10: 0.030976
Recall@10: 0.202025
MAP@10: 0.083306
NDCG@10: 0.120246
Coverage@10: 0.487434
Precision@20: 0.023173
Recall@20: 0.297631
MAP@20: 0.090996
NDCG@20: 0.147223
Coverage@20: 0.610075

BEST ALS - VALIDATION HIGH INTENT
Precision@5: 0.034248
Recall@5: 0.135462
MAP@5: 0.076266
NDCG@5: 0.094882
Coverage@5: 0.194853
Precision@10: 0.025265
Recall@10: 0.196525
MAP@10: 0.084815
NDCG@10: 0.115658
Coverage@10: 0.298178
Precision@20: 0.018031
Recall@20: 0.272887
MAP@20: 0.090447
NDCG@20: 0.136211
Coverage@20: 0.433933


In [24]:
# =========================
# Save Best ALS model and validation results
# =========================

import joblib
import json

best_als_model_path = MODELS_DIR / "best_als_model.pkl"
best_als_params_path = MODELS_DIR / "best_als_params.json"
best_als_results_path = REPORTS_DIR / "best_als_validation_results.csv"

# Save model
joblib.dump(best_als_model, best_als_model_path)

# Save best params
best_als_params = {
    "model": "ALS",
    "factors": BEST_ALS_FACTORS,
    "regularization": BEST_ALS_REGULARIZATION,
    "alpha": BEST_ALS_ALPHA,
    "iterations": BEST_ALS_ITERATIONS
}

with open(best_als_params_path, "w") as f:
    json.dump(best_als_params, f, indent=4)

# Save validation results
best_als_validation_all_row = {
    "model": "Best_ALS",
    "evaluation_type": "validation_all",
    **best_als_params,
    **best_als_results_validation_all
}

best_als_validation_high_intent_row = {
    "model": "Best_ALS",
    "evaluation_type": "validation_high_intent",
    **best_als_params,
    **best_als_results_validation_high_intent
}

best_als_results_df = pd.DataFrame([
    best_als_validation_all_row,
    best_als_validation_high_intent_row
])

best_als_results_df.to_csv(best_als_results_path, index=False)

print("Saved Best ALS model to:")
print(best_als_model_path)

print("\nSaved Best ALS params to:")
print(best_als_params_path)

print("\nSaved Best ALS validation results to:")
print(best_als_results_path)

display(best_als_results_df)

Saved Best ALS model to:
C:\Users\pc\Documents\recommender_project\trial two\models\best_als_model.pkl

Saved Best ALS params to:
C:\Users\pc\Documents\recommender_project\trial two\models\best_als_params.json

Saved Best ALS validation results to:
C:\Users\pc\Documents\recommender_project\trial two\reports\best_als_validation_results.csv


,model,evaluation_type,factors,regularization,alpha,iterations,Precision@5,Recall@5,MAP@5,NDCG@5,...,Precision@10,Recall@10,MAP@10,NDCG@10,Coverage@10,Precision@20,Recall@20,MAP@20,NDCG@20,Coverage@20
0,ALS,validation_all,192,0.05,80,20,0.039731,0.132181,0.073201,0.095338,...,0.030976,0.202025,0.083306,0.120246,0.487434,0.023173,0.297631,0.090996,0.147223,0.610075
1,ALS,validation_high_intent,192,0.05,80,20,0.034248,0.135462,0.076266,0.094882,...,0.025265,0.196525,0.084815,0.115658,0.298178,0.018031,0.272887,0.090447,0.136211,0.433933


In [26]:
# =========================
# Check 2 - Recommendation leakage checks
# =========================

def check_recommendation_sanity(recommendations, user_seen_items_idx, num_items, name="model"):
    """
    Check recommendation output quality:
    - no already-seen items
    - no invalid item indices
    - no duplicate recommendations per user
    - recommendation list length is valid
    """

    print("=" * 60)
    print(f"CHECK 2: RECOMMENDATION SANITY - {name}")
    print("=" * 60)

    users_checked = 0
    seen_item_leaks = 0
    invalid_items = 0
    duplicate_recommendation_users = 0
    empty_recommendation_users = 0

    recommendation_lengths = []

    for user_idx, recs in recommendations.items():
        users_checked += 1
        recommendation_lengths.append(len(recs))

        seen_items = set(user_seen_items_idx.get(user_idx, []))

        # Seen item leakage
        leaked_items = set(recs) & seen_items
        if len(leaked_items) > 0:
            seen_item_leaks += 1

        # Invalid item ids
        invalid = [item for item in recs if item < 0 or item >= num_items]
        if len(invalid) > 0:
            invalid_items += 1

        # Duplicates
        if len(recs) != len(set(recs)):
            duplicate_recommendation_users += 1

        # Empty recs
        if len(recs) == 0:
            empty_recommendation_users += 1

    print("Users checked:", users_checked)
    print("Users with already-seen item leakage:", seen_item_leaks)
    print("Users with invalid item ids:", invalid_items)
    print("Users with duplicate recommendations:", duplicate_recommendation_users)
    print("Users with empty recommendations:", empty_recommendation_users)

    print("\nRecommendation length summary:")
    print(pd.Series(recommendation_lengths).describe())

    total_issues = (
        seen_item_leaks
        + invalid_items
        + duplicate_recommendation_users
        + empty_recommendation_users
    )

    print("\n" + "-" * 60)

    if total_issues == 0:
        print("PASS: Recommendation outputs look valid.")
    else:
        print("WARNING: Recommendation issues detected.")

    return total_issues


als_recommendation_issues = check_recommendation_sanity(
    recommendations=best_als_recs_validation_high_intent,
    user_seen_items_idx=user_seen_items_idx,
    num_items=train_user_item_matrix.shape[1],
    name="Best ALS - Validation High Intent"
)

CHECK 2: RECOMMENDATION SANITY - Best ALS - Validation High Intent
Users checked: 2260
Users with already-seen item leakage: 0
Users with invalid item ids: 0
Users with duplicate recommendations: 0
Users with empty recommendations: 0

Recommendation length summary:
count    2260.0
mean       20.0
std         0.0
min        20.0
25%        20.0
50%        20.0
75%        20.0
max        20.0
dtype: float64

------------------------------------------------------------
PASS: Recommendation outputs look valid.


In [27]:
# =========================
# Check 3 - Compare Best ALS with Popularity Baseline
# =========================

comparison_rows = [
    {
        "model": "Popularity",
        "evaluation_type": "validation_high_intent",
        "Precision@10": popularity_results_validation_high_intent["Precision@10"],
        "Recall@10": popularity_results_validation_high_intent["Recall@10"],
        "MAP@10": popularity_results_validation_high_intent["MAP@10"],
        "NDCG@10": popularity_results_validation_high_intent["NDCG@10"],
        "Coverage@10": popularity_results_validation_high_intent["Coverage@10"],
    },
    {
        "model": "Best ALS",
        "evaluation_type": "validation_high_intent",
        "Precision@10": best_als_results_validation_high_intent["Precision@10"],
        "Recall@10": best_als_results_validation_high_intent["Recall@10"],
        "MAP@10": best_als_results_validation_high_intent["MAP@10"],
        "NDCG@10": best_als_results_validation_high_intent["NDCG@10"],
        "Coverage@10": best_als_results_validation_high_intent["Coverage@10"],
    }
]

baseline_comparison_df = pd.DataFrame(comparison_rows)

display(baseline_comparison_df)

ndcg_improvement = (
    best_als_results_validation_high_intent["NDCG@10"]
    / popularity_results_validation_high_intent["NDCG@10"]
)

recall_improvement = (
    best_als_results_validation_high_intent["Recall@10"]
    / popularity_results_validation_high_intent["Recall@10"]
)

coverage_improvement = (
    best_als_results_validation_high_intent["Coverage@10"]
    / popularity_results_validation_high_intent["Coverage@10"]
)

print(f"NDCG@10 improvement over Popularity: {ndcg_improvement:.2f}x")
print(f"Recall@10 improvement over Popularity: {recall_improvement:.2f}x")
print(f"Coverage@10 improvement over Popularity: {coverage_improvement:.2f}x")

,model,evaluation_type,Precision@10,Recall@10,MAP@10,NDCG@10,Coverage@10
0,Popularity,validation_high_intent,0.003584,0.023781,0.011498,0.015406,0.001317
1,Best ALS,validation_high_intent,0.025265,0.196525,0.084815,0.115658,0.298178


NDCG@10 improvement over Popularity: 7.51x
Recall@10 improvement over Popularity: 8.26x
Coverage@10 improvement over Popularity: 226.42x


In [28]:
# =========================
# BPR - Import
# =========================

from implicit.bpr import BayesianPersonalizedRanking

print("BPR imported successfully.")

BPR imported successfully.


In [29]:
# =========================
# BPR - Train initial model
# =========================

BPR_FACTORS = 64
BPR_LEARNING_RATE = 0.05
BPR_REGULARIZATION = 0.01
BPR_ITERATIONS = 100

bpr_model = BayesianPersonalizedRanking(
    factors=BPR_FACTORS,
    learning_rate=BPR_LEARNING_RATE,
    regularization=BPR_REGULARIZATION,
    iterations=BPR_ITERATIONS,
    random_state=42,
    verify_negative_samples=True
)

bpr_model.fit(train_user_item_matrix)

print("BPR model trained successfully.")
print("Factors:", BPR_FACTORS)
print("Learning rate:", BPR_LEARNING_RATE)
print("Regularization:", BPR_REGULARIZATION)
print("Iterations:", BPR_ITERATIONS)

  0%|          | 0/100 [00:00<?, ?it/s]

BPR model trained successfully.
Factors: 64
Learning rate: 0.05
Regularization: 0.01
Iterations: 100


In [30]:
# =========================
# BPR - Generate recommendations
# =========================

def generate_bpr_recommendations(model, user_ids, user_item_matrix, k=20):
    """
    Generate BPR recommendations for encoded user_idx values.
    """
    recommendations = {}

    for user_idx in user_ids:
        recommended_items, scores = model.recommend(
            userid=user_idx,
            user_items=user_item_matrix[user_idx],
            N=k,
            filter_already_liked_items=True
        )

        recommendations[user_idx] = recommended_items.tolist()

    return recommendations


print("BPR recommendation function is ready.")

BPR recommendation function is ready.


In [31]:
# =========================
# BPR - Evaluate on validation all + high-intent
# =========================

NUM_ITEMS = train_user_item_matrix.shape[1]
K_MAX = 20

# Validation ALL
validation_users_all = list(validation_gt_all.keys())

bpr_recs_validation_all = generate_bpr_recommendations(
    model=bpr_model,
    user_ids=validation_users_all,
    user_item_matrix=train_user_item_matrix,
    k=K_MAX
)

bpr_results_validation_all = evaluate_recommendations(
    recommendations=bpr_recs_validation_all,
    ground_truth=validation_gt_all,
    num_items=NUM_ITEMS,
    k_values=[5, 10, 20]
)

# Validation HIGH INTENT
validation_users_high_intent = list(validation_gt_high_intent.keys())

bpr_recs_validation_high_intent = generate_bpr_recommendations(
    model=bpr_model,
    user_ids=validation_users_high_intent,
    user_item_matrix=train_user_item_matrix,
    k=K_MAX
)

bpr_results_validation_high_intent = evaluate_recommendations(
    recommendations=bpr_recs_validation_high_intent,
    ground_truth=validation_gt_high_intent,
    num_items=NUM_ITEMS,
    k_values=[5, 10, 20]
)

print_results(
    "BPR - VALIDATION ALL",
    bpr_results_validation_all
)

print_results(
    "BPR - VALIDATION HIGH INTENT",
    bpr_results_validation_high_intent
)


BPR - VALIDATION ALL
Precision@5: 0.024172
Recall@5: 0.081607
MAP@5: 0.044519
NDCG@5: 0.058444
Coverage@5: 0.673837
Precision@10: 0.019424
Recall@10: 0.128464
MAP@10: 0.051085
NDCG@10: 0.075093
Coverage@10: 0.795050
Precision@20: 0.014834
Recall@20: 0.191255
MAP@20: 0.056016
NDCG@20: 0.092900
Coverage@20: 0.883505

BPR - VALIDATION HIGH INTENT
Precision@5: 0.017434
Recall@5: 0.071051
MAP@5: 0.039863
NDCG@5: 0.049390
Coverage@5: 0.306190
Precision@10: 0.013053
Recall@10: 0.103625
MAP@10: 0.044179
NDCG@10: 0.060521
Coverage@10: 0.460656
Precision@20: 0.010509
Recall@20: 0.161241
MAP@20: 0.048375
NDCG@20: 0.076010
Coverage@20: 0.637017


In [32]:
# =========================
# BPR - Small hyperparameter tuning
# =========================

bpr_tuning_configs = [
    {"factors": 64, "learning_rate": 0.05, "regularization": 0.01, "iterations": 100},
    {"factors": 128, "learning_rate": 0.05, "regularization": 0.01, "iterations": 100},
    {"factors": 192, "learning_rate": 0.05, "regularization": 0.01, "iterations": 100},
    {"factors": 128, "learning_rate": 0.03, "regularization": 0.01, "iterations": 100},
    {"factors": 128, "learning_rate": 0.05, "regularization": 0.05, "iterations": 100},
    {"factors": 128, "learning_rate": 0.05, "regularization": 0.01, "iterations": 150},
]

print("Number of BPR configs:", len(bpr_tuning_configs))
bpr_tuning_configs

Number of BPR configs: 6


[{'factors': 64,
  'learning_rate': 0.05,
  'regularization': 0.01,
  'iterations': 100},
 {'factors': 128,
  'learning_rate': 0.05,
  'regularization': 0.01,
  'iterations': 100},
 {'factors': 192,
  'learning_rate': 0.05,
  'regularization': 0.01,
  'iterations': 100},
 {'factors': 128,
  'learning_rate': 0.03,
  'regularization': 0.01,
  'iterations': 100},
 {'factors': 128,
  'learning_rate': 0.05,
  'regularization': 0.05,
  'iterations': 100},
 {'factors': 128,
  'learning_rate': 0.05,
  'regularization': 0.01,
  'iterations': 150}]

In [37]:
# =========================
# BPR - Tune on validation high-intent
# =========================

bpr_tuning_results = []

validation_users_hi = list(validation_gt_high_intent.keys())
NUM_ITEMS = train_user_item_matrix.shape[1]
K_MAX = 20

for i, config in enumerate(bpr_tuning_configs, start=1):
    print("\n" + "=" * 70)
    print(f"Training BPR config {i}/{len(bpr_tuning_configs)}")
    print(config)
    print("=" * 70)

    model = BayesianPersonalizedRanking(
        factors=config["factors"],
        learning_rate=config["learning_rate"],
        regularization=config["regularization"],
        iterations=config["iterations"],
        random_state=42,
        verify_negative_samples=True
    )

    model.fit(train_user_item_matrix)

    recs_hi = generate_bpr_recommendations(
        model=model,
        user_ids=validation_users_hi,
        user_item_matrix=train_user_item_matrix,
        k=K_MAX
    )

    results_hi = evaluate_recommendations(
        recommendations=recs_hi,
        ground_truth=validation_gt_high_intent,
        num_items=NUM_ITEMS,
        k_values=[5, 10, 20]
    )

    row = {
        "model": "BPR",
        **config,
        **results_hi
    }

    bpr_tuning_results.append(row)

    print("Precision@10:", round(results_hi["Precision@10"], 6))
    print("Recall@10:", round(results_hi["Recall@10"], 6))
    print("MAP@10:", round(results_hi["MAP@10"], 6))
    print("NDCG@10:", round(results_hi["NDCG@10"], 6))
    print("Coverage@10:", round(results_hi["Coverage@10"], 6))


bpr_tuning_results_df = pd.DataFrame(bpr_tuning_results)

bpr_tuning_results_df = bpr_tuning_results_df.sort_values(
    by="NDCG@10",
    ascending=False
).reset_index(drop=True)

display(bpr_tuning_results_df)

bpr_tuning_results_path = REPORTS_DIR / "bpr_tuning_validation_high_intent.csv"
bpr_tuning_results_df.to_csv(bpr_tuning_results_path, index=False)

print("Saved BPR tuning results to:")
print(bpr_tuning_results_path)


Training BPR config 1/6
{'factors': 64, 'learning_rate': 0.05, 'regularization': 0.01, 'iterations': 100}


  0%|          | 0/100 [00:00<?, ?it/s]

Precision@10: 0.013186
Recall@10: 0.104104
MAP@10: 0.044381
NDCG@10: 0.060812
Coverage@10: 0.460656

Training BPR config 2/6
{'factors': 128, 'learning_rate': 0.05, 'regularization': 0.01, 'iterations': 100}


  0%|          | 0/100 [00:00<?, ?it/s]

Precision@10: 0.012389
Recall@10: 0.098462
MAP@10: 0.040798
NDCG@10: 0.056833
Coverage@10: 0.460217

Training BPR config 3/6
{'factors': 192, 'learning_rate': 0.05, 'regularization': 0.01, 'iterations': 100}


...

Precision@10: 0.011239
Recall@10: 0.088599
MAP@10: 0.033852
NDCG@10: 0.048737
Coverage@10: 0.354368

Training BPR config 6/6
{'factors': 128, 'learning_rate': 0.05, 'regularization': 0.01, 'iterations': 150}


  0%|          | 0/150 [00:00<?, ?it/s]

Precision@10: 0.013496
Recall@10: 0.106493
MAP@10: 0.041555
NDCG@10: 0.059568
Coverage@10: 0.497092


,model,factors,learning_rate,regularization,iterations,Precision@5,Recall@5,MAP@5,NDCG@5,Coverage@5,Precision@10,Recall@10,MAP@10,NDCG@10,Coverage@10,Precision@20,Recall@20,MAP@20,NDCG@20,Coverage@20
0,BPR,64,0.05,0.01,100,0.017434,0.071493,0.040029,0.049535,0.307561,0.013186,0.104104,0.044381,0.060812,0.460656,0.010420,0.160282,0.048497,0.075859,0.637072
1,BPR,128,0.05,0.01,150,0.017876,0.070866,0.036673,0.047444,0.325615,0.013496,0.106493,0.041555,0.059568,0.497092,0.010642,0.163739,0.045856,0.074999,0.685744
2,BPR,192,0.05,0.01,100,0.017168,0.068580,0.037616,0.047469,0.303720,0.012876,0.100524,0.041666,0.058227,0.459449,0.009867,0.153869,0.045568,0.072361,0.635810
3,BPR,128,0.05,0.01,100,0.016018,0.064583,0.036172,0.045242,0.302897,0.012389,0.098462,0.040798,0.056833,0.460217,0.009934,0.152979,0.044932,0.071637,0.633341
4,BPR,128,0.03,0.01,100,0.015133,0.060343,0.031670,0.040653,0.258999,0.011814,0.094469,0.036261,0.052161,0.383889,0.008960,0.139187,0.039693,0.064342,0.526229
5,BPR,128,0.05,0.05,100,0.013805,0.054071,0.029124,0.037105,0.228874,0.011239,0.088599,0.033852,0.048737,0.354368,0.009270,0.145533,0.038085,0.063986,0.521894


Saved BPR tuning results to:
C:\Users\pc\Documents\recommender_project\trial two\reports\bpr_tuning_validation_high_intent.csv


In [38]:
# =========================
# Train and save Best BPR model
# =========================

BEST_BPR_FACTORS = 64
BEST_BPR_LEARNING_RATE = 0.05
BEST_BPR_REGULARIZATION = 0.01
BEST_BPR_ITERATIONS = 100

best_bpr_model = BayesianPersonalizedRanking(
    factors=BEST_BPR_FACTORS,
    learning_rate=BEST_BPR_LEARNING_RATE,
    regularization=BEST_BPR_REGULARIZATION,
    iterations=BEST_BPR_ITERATIONS,
    random_state=42,
    verify_negative_samples=True
)

best_bpr_model.fit(train_user_item_matrix)

print("Best BPR model trained successfully.")
print("factors:", BEST_BPR_FACTORS)
print("learning_rate:", BEST_BPR_LEARNING_RATE)
print("regularization:", BEST_BPR_REGULARIZATION)
print("iterations:", BEST_BPR_ITERATIONS)

  0%|          | 0/100 [00:00<?, ?it/s]

Best BPR model trained successfully.
factors: 64
learning_rate: 0.05
regularization: 0.01
iterations: 100


In [39]:
# =========================
# Evaluate Best BPR on validation all + high-intent
# =========================

NUM_ITEMS = train_user_item_matrix.shape[1]
K_MAX = 20

# Validation ALL
validation_users_all = list(validation_gt_all.keys())

best_bpr_recs_validation_all = generate_bpr_recommendations(
    model=best_bpr_model,
    user_ids=validation_users_all,
    user_item_matrix=train_user_item_matrix,
    k=K_MAX
)

best_bpr_results_validation_all = evaluate_recommendations(
    recommendations=best_bpr_recs_validation_all,
    ground_truth=validation_gt_all,
    num_items=NUM_ITEMS,
    k_values=[5, 10, 20]
)

# Validation HIGH INTENT
validation_users_high_intent = list(validation_gt_high_intent.keys())

best_bpr_recs_validation_high_intent = generate_bpr_recommendations(
    model=best_bpr_model,
    user_ids=validation_users_high_intent,
    user_item_matrix=train_user_item_matrix,
    k=K_MAX
)

best_bpr_results_validation_high_intent = evaluate_recommendations(
    recommendations=best_bpr_recs_validation_high_intent,
    ground_truth=validation_gt_high_intent,
    num_items=NUM_ITEMS,
    k_values=[5, 10, 20]
)

print_results(
    "BEST BPR - VALIDATION ALL",
    best_bpr_results_validation_all
)

print_results(
    "BEST BPR - VALIDATION HIGH INTENT",
    best_bpr_results_validation_high_intent
)


BEST BPR - VALIDATION ALL
Precision@5: 0.024155
Recall@5: 0.081542
MAP@5: 0.044302
NDCG@5: 0.058213
Coverage@5: 0.671971
Precision@10: 0.019345
Recall@10: 0.127791
MAP@10: 0.050793
NDCG@10: 0.074674
Coverage@10: 0.794831
Precision@20: 0.014874
Recall@20: 0.191566
MAP@20: 0.055806
NDCG@20: 0.092772
Coverage@20: 0.882902

BEST BPR - VALIDATION HIGH INTENT
Precision@5: 0.017168
Recall@5: 0.070387
MAP@5: 0.039171
NDCG@5: 0.048618
Coverage@5: 0.305806
Precision@10: 0.013142
Recall@10: 0.104546
MAP@10: 0.043731
NDCG@10: 0.060335
Coverage@10: 0.460821
Precision@20: 0.010442
Recall@20: 0.160031
MAP@20: 0.047775
NDCG@20: 0.075278
Coverage@20: 0.636962


In [40]:
# =========================
# Save Best BPR model and validation results
# =========================

best_bpr_model_path = MODELS_DIR / "best_bpr_model.pkl"
best_bpr_params_path = MODELS_DIR / "best_bpr_params.json"
best_bpr_results_path = REPORTS_DIR / "best_bpr_validation_results.csv"

# Save model
joblib.dump(best_bpr_model, best_bpr_model_path)

# Save params
best_bpr_params = {
    "model": "BPR",
    "factors": BEST_BPR_FACTORS,
    "learning_rate": BEST_BPR_LEARNING_RATE,
    "regularization": BEST_BPR_REGULARIZATION,
    "iterations": BEST_BPR_ITERATIONS
}

with open(best_bpr_params_path, "w") as f:
    json.dump(best_bpr_params, f, indent=4)

# Save validation results
best_bpr_validation_all_row = {
    "model": "Best_BPR",
    "evaluation_type": "validation_all",
    **best_bpr_params,
    **best_bpr_results_validation_all
}

best_bpr_validation_high_intent_row = {
    "model": "Best_BPR",
    "evaluation_type": "validation_high_intent",
    **best_bpr_params,
    **best_bpr_results_validation_high_intent
}

best_bpr_results_df = pd.DataFrame([
    best_bpr_validation_all_row,
    best_bpr_validation_high_intent_row
])

best_bpr_results_df.to_csv(best_bpr_results_path, index=False)

print("Saved Best BPR model to:")
print(best_bpr_model_path)

print("\nSaved Best BPR params to:")
print(best_bpr_params_path)

print("\nSaved Best BPR validation results to:")
print(best_bpr_results_path)

display(best_bpr_results_df)

Saved Best BPR model to:
C:\Users\pc\Documents\recommender_project\trial two\models\best_bpr_model.pkl

Saved Best BPR params to:
C:\Users\pc\Documents\recommender_project\trial two\models\best_bpr_params.json

Saved Best BPR validation results to:
C:\Users\pc\Documents\recommender_project\trial two\reports\best_bpr_validation_results.csv


,model,evaluation_type,factors,learning_rate,regularization,iterations,Precision@5,Recall@5,MAP@5,NDCG@5,...,Precision@10,Recall@10,MAP@10,NDCG@10,Coverage@10,Precision@20,Recall@20,MAP@20,NDCG@20,Coverage@20
0,BPR,validation_all,64,0.05,0.01,100,0.024155,0.081542,0.044302,0.058213,...,0.019345,0.127791,0.050793,0.074674,0.794831,0.014874,0.191566,0.055806,0.092772,0.882902
1,BPR,validation_high_intent,64,0.05,0.01,100,0.017168,0.070387,0.039171,0.048618,...,0.013142,0.104546,0.043731,0.060335,0.460821,0.010442,0.160031,0.047775,0.075278,0.636962


In [ ]:
#ALS is the strongest model so far for recommendation accuracy.
#BPR provides better catalog coverage but lower ranking quality.

In [41]:
# =========================
# BPR sanity check
# =========================

bpr_recommendation_issues = check_recommendation_sanity(
    recommendations=best_bpr_recs_validation_high_intent,
    user_seen_items_idx=user_seen_items_idx,
    num_items=train_user_item_matrix.shape[1],
    name="Best BPR - Validation High Intent"
)

CHECK 2: RECOMMENDATION SANITY - Best BPR - Validation High Intent
Users checked: 2260
Users with already-seen item leakage: 0
Users with invalid item ids: 0
Users with duplicate recommendations: 0
Users with empty recommendations: 0

Recommendation length summary:
count    2260.0
mean       20.0
std         0.0
min        20.0
25%        20.0
50%        20.0
75%        20.0
max        20.0
dtype: float64

------------------------------------------------------------
PASS: Recommendation outputs look valid.


In [42]:
# =========================
# Content-Based - Build item content matrix
# =========================

from scipy.sparse import hstack, csr_matrix
from sklearn.preprocessing import OneHotEncoder, normalize

# Check alignment first
print("Item features shape:", item_features.shape)
print("TF-IDF matrix shape:", item_tfidf_matrix.shape)

print("\nItem idx check:")
print("Min item_idx:", item_features["item_idx"].min())
print("Max item_idx:", item_features["item_idx"].max())
print("Expected max:", train_user_item_matrix.shape[1] - 1)

# One-hot encode category and availability
try:
    category_encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=True)
    availability_encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=True)
except TypeError:
    category_encoder = OneHotEncoder(handle_unknown="ignore", sparse=True)
    availability_encoder = OneHotEncoder(handle_unknown="ignore", sparse=True)

category_matrix = category_encoder.fit_transform(
    item_features[["categoryid"]].astype(str)
)

availability_matrix = availability_encoder.fit_transform(
    item_features[["available"]].astype(str)
)

# Give category a little more importance
CATEGORY_WEIGHT = 2.0
AVAILABILITY_WEIGHT = 0.5
TFIDF_WEIGHT = 1.0

item_content_matrix = hstack([
    item_tfidf_matrix * TFIDF_WEIGHT,
    category_matrix * CATEGORY_WEIGHT,
    availability_matrix * AVAILABILITY_WEIGHT
]).tocsr()

# Normalize item vectors for cosine similarity
item_content_matrix = normalize(item_content_matrix, norm="l2", axis=1)

print("\nContent matrix created.")
print("Item content matrix shape:", item_content_matrix.shape)
print("Non-zero values:", item_content_matrix.nnz)

Item features shape: (18224, 5)
TF-IDF matrix shape: (18224, 300)

Item idx check:
Min item_idx: 0
Max item_idx: 18223
Expected max: 18223

Content matrix created.
Item content matrix shape: (18224, 1027)
Non-zero values: 374064


In [43]:
# =========================
# Content-Based - Recommendation function
# =========================

def build_user_profile(user_idx, train_df, item_content_matrix):
    """
    Build a content-based user profile from user's training interactions.
    """
    user_history = train_df[train_df["user_idx"] == user_idx]

    if user_history.empty:
        return None

    item_indices = user_history["item_idx"].values
    weights = user_history["interaction_strength"].astype("float32").values

    # Avoid division by zero
    if weights.sum() == 0:
        weights = np.ones_like(weights)

    user_item_vectors = item_content_matrix[item_indices]

    # Weighted average of item vectors
    user_profile = user_item_vectors.multiply(weights.reshape(-1, 1)).sum(axis=0)
    user_profile = np.asarray(user_profile)

    user_profile = csr_matrix(user_profile)
    user_profile = normalize(user_profile, norm="l2", axis=1)

    return user_profile


def generate_content_based_recommendations(
    users,
    train_df,
    item_content_matrix,
    user_seen_items_idx,
    k=20
):
    """
    Generate content-based recommendations for users.
    """
    recommendations = {}

    num_items = item_content_matrix.shape[0]

    for user_idx in users:
        user_profile = build_user_profile(
            user_idx=user_idx,
            train_df=train_df,
            item_content_matrix=item_content_matrix
        )

        if user_profile is None:
            recommendations[user_idx] = []
            continue

        # Cosine similarity because vectors are normalized
        scores = user_profile.dot(item_content_matrix.T).toarray().ravel()

        # Filter already-seen items
        seen_items = user_seen_items_idx.get(user_idx, [])
        scores[seen_items] = -np.inf

        # Get top-K item indices
        if k < num_items:
            top_items = np.argpartition(-scores, kth=k)[:k]
            top_items = top_items[np.argsort(-scores[top_items])]
        else:
            top_items = np.argsort(-scores)

        recommendations[user_idx] = top_items.tolist()

    return recommendations


print("Content-based recommendation function is ready.")

Content-based recommendation function is ready.


In [45]:
# =========================
# Content-Based - Quick sample evaluation
# =========================

NUM_ITEMS = train_user_item_matrix.shape[1]
K_MAX = 20

# Use small samples first
sample_validation_users_all = list(validation_gt_all.keys())[:1000]
sample_validation_users_high_intent = list(validation_gt_high_intent.keys())[:500]

# Filter ground truth to sampled users
sample_validation_gt_all = {
    user: validation_gt_all[user]
    for user in sample_validation_users_all
}

sample_validation_gt_high_intent = {
    user: validation_gt_high_intent[user]
    for user in sample_validation_users_high_intent
}

# Generate recommendations for sample
content_recs_sample_all = generate_content_based_recommendations(
    users=sample_validation_users_all,
    train_df=train_interactions,
    item_content_matrix=item_content_matrix,
    user_seen_items_idx=user_seen_items_idx,
    k=K_MAX
)

content_results_sample_all = evaluate_recommendations(
    recommendations=content_recs_sample_all,
    ground_truth=sample_validation_gt_all,
    num_items=NUM_ITEMS,
    k_values=[5, 10, 20]
)

content_recs_sample_high_intent = generate_content_based_recommendations(
    users=sample_validation_users_high_intent,
    train_df=train_interactions,
    item_content_matrix=item_content_matrix,
    user_seen_items_idx=user_seen_items_idx,
    k=K_MAX
)

content_results_sample_high_intent = evaluate_recommendations(
    recommendations=content_recs_sample_high_intent,
    ground_truth=sample_validation_gt_high_intent,
    num_items=NUM_ITEMS,
    k_values=[5, 10, 20]
)

print_results(
    "CONTENT-BASED SAMPLE - VALIDATION ALL",
    content_results_sample_all
)

print_results(
    "CONTENT-BASED SAMPLE - VALIDATION HIGH INTENT",
    content_results_sample_high_intent
)


CONTENT-BASED SAMPLE - VALIDATION ALL
Precision@5: 0.030000
Recall@5: 0.100876
MAP@5: 0.052838
NDCG@5: 0.070411
Coverage@5: 0.198968
Precision@10: 0.024800
Recall@10: 0.169669
MAP@10: 0.062947
NDCG@10: 0.094420
Coverage@10: 0.330169
Precision@20: 0.019100
Recall@20: 0.262279
MAP@20: 0.070252
NDCG@20: 0.119945
Coverage@20: 0.504006

CONTENT-BASED SAMPLE - VALIDATION HIGH INTENT
Precision@5: 0.026000
Recall@5: 0.110500
MAP@5: 0.062250
NDCG@5: 0.076310
Coverage@5: 0.115946
Precision@10: 0.017600
Recall@10: 0.143667
MAP@10: 0.067136
NDCG@10: 0.088179
Coverage@10: 0.206925
Precision@20: 0.012400
Recall@20: 0.198119
MAP@20: 0.071246
NDCG@20: 0.102758
Coverage@20: 0.346741


In [47]:
# =========================
# Content-Based - Build all user profiles fast
# =========================

from sklearn.preprocessing import normalize

# Build user content profiles from train interactions
# Shape: users x content_features
user_content_profiles = train_user_item_matrix.dot(item_content_matrix)

# Normalize user profiles for cosine similarity
user_content_profiles = normalize(user_content_profiles, norm="l2", axis=1)

print("User content profiles created.")
print("User content profiles shape:", user_content_profiles.shape)
print("Item content matrix shape:", item_content_matrix.shape)

User content profiles created.
User content profiles shape: (22890, 1027)
Item content matrix shape: (18224, 1027)


In [48]:
# =========================
# Content-Based - Fast batch recommendation function
# =========================

def generate_content_based_recommendations_fast(
    users,
    user_content_profiles,
    item_content_matrix,
    user_seen_items_idx,
    k=20,
    batch_size=500
):
    """
    Generate content-based recommendations using batch matrix multiplication.
    """
    recommendations = {}
    num_items = item_content_matrix.shape[0]

    users = list(users)

    for start in range(0, len(users), batch_size):
        batch_users = users[start:start + batch_size]

        # User profiles for this batch
        batch_profiles = user_content_profiles[batch_users]

        # Scores = cosine similarity because both matrices are normalized
        score_matrix = batch_profiles.dot(item_content_matrix.T).toarray()

        # Filter already-seen items
        for row_idx, user_idx in enumerate(batch_users):
            seen_items = user_seen_items_idx.get(user_idx, [])
            score_matrix[row_idx, seen_items] = -np.inf

        # Top-K recommendations
        for row_idx, user_idx in enumerate(batch_users):
            scores = score_matrix[row_idx]

            if k < num_items:
                top_items = np.argpartition(-scores, kth=k)[:k]
                top_items = top_items[np.argsort(-scores[top_items])]
            else:
                top_items = np.argsort(-scores)

            recommendations[user_idx] = top_items.tolist()

        print(f"Processed users {start + len(batch_users):,}/{len(users):,}")

    return recommendations


print("Fast content-based recommendation function is ready.")

Fast content-based recommendation function is ready.


In [49]:
# =========================
# Content-Based - Full validation high-intent evaluation
# =========================

NUM_ITEMS = train_user_item_matrix.shape[1]
K_MAX = 20

validation_users_high_intent = list(validation_gt_high_intent.keys())

content_recs_validation_high_intent = generate_content_based_recommendations_fast(
    users=validation_users_high_intent,
    user_content_profiles=user_content_profiles,
    item_content_matrix=item_content_matrix,
    user_seen_items_idx=user_seen_items_idx,
    k=K_MAX,
    batch_size=500
)

content_results_validation_high_intent = evaluate_recommendations(
    recommendations=content_recs_validation_high_intent,
    ground_truth=validation_gt_high_intent,
    num_items=NUM_ITEMS,
    k_values=[5, 10, 20]
)

print_results(
    "CONTENT-BASED - VALIDATION HIGH INTENT",
    content_results_validation_high_intent
)

Processed users 500/2,260
Processed users 1,000/2,260
Processed users 1,500/2,260
Processed users 2,000/2,260
Processed users 2,260/2,260

CONTENT-BASED - VALIDATION HIGH INTENT
Precision@5: 0.022743
Recall@5: 0.093916
MAP@5: 0.051889
NDCG@5: 0.064541
Coverage@5: 0.351405
Precision@10: 0.018097
Recall@10: 0.145280
MAP@10: 0.059287
NDCG@10: 0.082292
Coverage@10: 0.525955
Precision@20: 0.013031
Recall@20: 0.204802
MAP@20: 0.063812
NDCG@20: 0.098355
Coverage@20: 0.713180


In [50]:
# =========================
# Content-Based - Sanity check
# =========================

content_recommendation_issues = check_recommendation_sanity(
    recommendations=content_recs_validation_high_intent,
    user_seen_items_idx=user_seen_items_idx,
    num_items=train_user_item_matrix.shape[1],
    name="Content-Based - Validation High Intent"
)

CHECK 2: RECOMMENDATION SANITY - Content-Based - Validation High Intent
Users checked: 2260
Users with already-seen item leakage: 0
Users with invalid item ids: 0
Users with duplicate recommendations: 0
Users with empty recommendations: 0

Recommendation length summary:
count    2260.0
mean       20.0
std         0.0
min        20.0
25%        20.0
50%        20.0
75%        20.0
max        20.0
dtype: float64

------------------------------------------------------------
PASS: Recommendation outputs look valid.


In [51]:
# =========================
# Content-Based - Tune feature weights
# =========================

content_weight_configs = [
    {"tfidf_weight": 1.0, "category_weight": 1.0, "availability_weight": 0.0},
    {"tfidf_weight": 1.0, "category_weight": 2.0, "availability_weight": 0.5},
    {"tfidf_weight": 1.0, "category_weight": 3.0, "availability_weight": 0.5},
    {"tfidf_weight": 2.0, "category_weight": 2.0, "availability_weight": 0.5},
    {"tfidf_weight": 2.0, "category_weight": 3.0, "availability_weight": 0.5},
]

content_tuning_results = []

validation_users_hi = list(validation_gt_high_intent.keys())
NUM_ITEMS = train_user_item_matrix.shape[1]
K_MAX = 20

for i, config in enumerate(content_weight_configs, start=1):
    print("\n" + "=" * 70)
    print(f"Content-Based config {i}/{len(content_weight_configs)}")
    print(config)
    print("=" * 70)

    # Build item content matrix with current weights
    tuned_item_content_matrix = hstack([
        item_tfidf_matrix * config["tfidf_weight"],
        category_matrix * config["category_weight"],
        availability_matrix * config["availability_weight"]
    ]).tocsr()

    tuned_item_content_matrix = normalize(
        tuned_item_content_matrix,
        norm="l2",
        axis=1
    )

    # Build user profiles
    tuned_user_content_profiles = train_user_item_matrix.dot(
        tuned_item_content_matrix
    )

    tuned_user_content_profiles = normalize(
        tuned_user_content_profiles,
        norm="l2",
        axis=1
    )

    # Generate recommendations
    tuned_recs_hi = generate_content_based_recommendations_fast(
        users=validation_users_hi,
        user_content_profiles=tuned_user_content_profiles,
        item_content_matrix=tuned_item_content_matrix,
        user_seen_items_idx=user_seen_items_idx,
        k=K_MAX,
        batch_size=500
    )

    # Evaluate
    tuned_results_hi = evaluate_recommendations(
        recommendations=tuned_recs_hi,
        ground_truth=validation_gt_high_intent,
        num_items=NUM_ITEMS,
        k_values=[5, 10, 20]
    )

    row = {
        "model": "Content-Based",
        **config,
        **tuned_results_hi
    }

    content_tuning_results.append(row)

    print("Precision@10:", round(tuned_results_hi["Precision@10"], 6))
    print("Recall@10:", round(tuned_results_hi["Recall@10"], 6))
    print("MAP@10:", round(tuned_results_hi["MAP@10"], 6))
    print("NDCG@10:", round(tuned_results_hi["NDCG@10"], 6))
    print("Coverage@10:", round(tuned_results_hi["Coverage@10"], 6))


content_tuning_results_df = pd.DataFrame(content_tuning_results)

content_tuning_results_df = content_tuning_results_df.sort_values(
    by="NDCG@10",
    ascending=False
).reset_index(drop=True)

display(content_tuning_results_df)

content_tuning_path = REPORTS_DIR / "content_based_tuning_validation_high_intent.csv"
content_tuning_results_df.to_csv(content_tuning_path, index=False)

print("Saved Content-Based tuning results to:")
print(content_tuning_path)


Content-Based config 1/5
{'tfidf_weight': 1.0, 'category_weight': 1.0, 'availability_weight': 0.0}
Processed users 500/2,260
Processed users 1,000/2,260
Processed users 1,500/2,260
Processed users 2,000/2,260
Processed users 2,260/2,260
Precision@10: 0.017434
Recall@10: 0.141287
MAP@10: 0.059199
NDCG@10: 0.081136
Coverage@10: 0.526449

Content-Based config 2/5
{'tfidf_weight': 1.0, 'category_weight': 2.0, 'availability_weight': 0.5}
Processed users 500/2,260
Processed users 1,000/2,260
Processed users 1,500/2,260
Processed users 2,000/2,260
Processed users 2,260/2,260
Precision@10: 0.018097
Recall@10: 0.14528
MAP@10: 0.059287
NDCG@10: 0.082292
Coverage@10: 0.525955

Content-Based config 3/5
{'tfidf_weight': 1.0, 'category_weight': 3.0, 'availability_weight': 0.5}
Processed users 500/2,260
Processed users 1,000/2,260
Processed users 1,500/2,260
Processed users 2,000/2,260
Processed users 2,260/2,260
Precision@10: 0.018053
Recall@10: 0.144838
MAP@10: 0.059594
NDCG@10: 0.082461
Coverage@

,model,tfidf_weight,category_weight,availability_weight,Precision@5,Recall@5,MAP@5,NDCG@5,Coverage@5,Precision@10,Recall@10,MAP@10,NDCG@10,Coverage@10,Precision@20,Recall@20,MAP@20,NDCG@20,Coverage@20
0,Content-Based,2.0,3.0,0.5,0.023363,0.097013,0.056154,0.068571,0.351460,0.018142,0.145122,0.063107,0.085284,0.522662,0.012854,0.202519,0.067494,0.100801,0.711260
1,Content-Based,2.0,2.0,0.5,0.023363,0.096640,0.055221,0.067806,0.348826,0.018009,0.143580,0.061981,0.084074,0.520632,0.012876,0.201560,0.066417,0.099780,0.710272
2,Content-Based,1.0,3.0,0.5,0.022920,0.094395,0.052302,0.065010,0.352063,0.018053,0.144838,0.059594,0.082461,0.526504,0.012942,0.204189,0.064085,0.098415,0.709723
3,Content-Based,1.0,2.0,0.5,0.022743,0.093916,0.051889,0.064541,0.351405,0.018097,0.145280,0.059287,0.082292,0.525955,0.013031,0.204802,0.063812,0.098355,0.713180
4,Content-Based,1.0,1.0,0.0,0.023097,0.096192,0.052920,0.065711,0.353600,0.017434,0.141287,0.059199,0.081136,0.526449,0.012677,0.202412,0.063730,0.097460,0.709669


Saved Content-Based tuning results to:
C:\Users\pc\Documents\recommender_project\trial two\reports\content_based_tuning_validation_high_intent.csv


In [52]:
# =========================
# Build Best Content-Based matrices
# =========================

BEST_TFIDF_WEIGHT = 2.0
BEST_CATEGORY_WEIGHT = 3.0
BEST_AVAILABILITY_WEIGHT = 0.5

best_item_content_matrix = hstack([
    item_tfidf_matrix * BEST_TFIDF_WEIGHT,
    category_matrix * BEST_CATEGORY_WEIGHT,
    availability_matrix * BEST_AVAILABILITY_WEIGHT
]).tocsr()

best_item_content_matrix = normalize(
    best_item_content_matrix,
    norm="l2",
    axis=1
)

best_user_content_profiles = train_user_item_matrix.dot(
    best_item_content_matrix
)

best_user_content_profiles = normalize(
    best_user_content_profiles,
    norm="l2",
    axis=1
)

print("Best Content-Based matrices created.")
print("Best item content matrix:", best_item_content_matrix.shape)
print("Best user content profiles:", best_user_content_profiles.shape)

Best Content-Based matrices created.
Best item content matrix: (18224, 1027)
Best user content profiles: (22890, 1027)


In [55]:
# =========================
# Evaluate Best Content-Based on validation all
# =========================

NUM_ITEMS = train_user_item_matrix.shape[1]
K_MAX = 20

validation_users_all = list(validation_gt_all.keys())

best_content_recs_validation_all = generate_content_based_recommendations_fast(
    users=validation_users_all,
    user_content_profiles=best_user_content_profiles,
    item_content_matrix=best_item_content_matrix,
    user_seen_items_idx=user_seen_items_idx,
    k=K_MAX,
    batch_size=500
)

best_content_results_validation_all = evaluate_recommendations(
    recommendations=best_content_recs_validation_all,
    ground_truth=validation_gt_all,
    num_items=NUM_ITEMS,
    k_values=[5, 10, 20]
)

print_results(
    "BEST CONTENT-BASED - VALIDATION ALL",
    best_content_results_validation_all
)

Processed users 500/22,869
Processed users 1,000/22,869
Processed users 1,500/22,869
Processed users 2,000/22,869
Processed users 2,500/22,869
Processed users 3,000/22,869
Processed users 3,500/22,869
Processed users 4,000/22,869
Processed users 4,500/22,869
Processed users 5,000/22,869
Processed users 5,500/22,869
Processed users 6,000/22,869
Processed users 6,500/22,869
Processed users 7,000/22,869
Processed users 7,500/22,869
Processed users 8,000/22,869
Processed users 8,500/22,869
Processed users 9,000/22,869
Processed users 9,500/22,869
Processed users 10,000/22,869
Processed users 10,500/22,869
Processed users 11,000/22,869
Processed users 11,500/22,869
Processed users 12,000/22,869
Processed users 12,500/22,869
Processed users 13,000/22,869
Processed users 13,500/22,869
Processed users 14,000/22,869
Processed users 14,500/22,869
Processed users 15,000/22,869
Processed users 15,500/22,869
Processed users 16,000/22,869
Processed users 16,500/22,869
Processed users 17,000/22,869
P

In [56]:
# =========================
# Save Best Content-Based validation results
# =========================

best_content_results_validation_high_intent = content_tuning_results_df.iloc[0].to_dict()

# Remove config/model columns from metric dict
metric_columns = [
    col for col in content_tuning_results_df.columns
    if "@" in col
]

best_content_high_intent_metrics = {
    col: best_content_results_validation_high_intent[col]
    for col in metric_columns
}

best_content_params = {
    "model": "Content-Based",
    "tfidf_weight": BEST_TFIDF_WEIGHT,
    "category_weight": BEST_CATEGORY_WEIGHT,
    "availability_weight": BEST_AVAILABILITY_WEIGHT
}

best_content_validation_all_row = {
    "model": "Best_Content_Based",
    "evaluation_type": "validation_all",
    **best_content_params,
    **best_content_results_validation_all
}

best_content_validation_high_intent_row = {
    "model": "Best_Content_Based",
    "evaluation_type": "validation_high_intent",
    **best_content_params,
    **best_content_high_intent_metrics
}

best_content_results_df = pd.DataFrame([
    best_content_validation_all_row,
    best_content_validation_high_intent_row
])

best_content_results_path = REPORTS_DIR / "best_content_based_validation_results.csv"
best_content_params_path = MODELS_DIR / "best_content_based_params.json"

best_content_results_df.to_csv(best_content_results_path, index=False)

with open(best_content_params_path, "w") as f:
    json.dump(best_content_params, f, indent=4)

print("Saved Best Content-Based results to:")
print(best_content_results_path)

print("\nSaved Best Content-Based params to:")
print(best_content_params_path)

display(best_content_results_df)

Saved Best Content-Based results to:
C:\Users\pc\Documents\recommender_project\trial two\reports\best_content_based_validation_results.csv

Saved Best Content-Based params to:
C:\Users\pc\Documents\recommender_project\trial two\models\best_content_based_params.json


,model,evaluation_type,tfidf_weight,category_weight,availability_weight,Precision@5,Recall@5,MAP@5,NDCG@5,Coverage@5,Precision@10,Recall@10,MAP@10,NDCG@10,Coverage@10,Precision@20,Recall@20,MAP@20,NDCG@20,Coverage@20
0,Content-Based,validation_all,2.0,3.0,0.5,0.029079,0.101530,0.054432,0.070869,0.813597,0.023154,0.158942,0.062760,0.091241,0.904357,0.017797,0.239922,0.069193,0.113888,0.959284
1,Content-Based,validation_high_intent,2.0,3.0,0.5,0.023363,0.097013,0.056154,0.068571,0.351460,0.018142,0.145122,0.063107,0.085284,0.522662,0.012854,0.202519,0.067494,0.100801,0.711260


In [58]:
# =========================
# Model Comparison Tables
# Validation All + Validation High Intent
# =========================

def extract_at_10_metrics(results_dict):
    """
    Extract @10 metrics from a model results dictionary.
    """
    return {
        "Precision@10": results_dict["Precision@10"],
        "Recall@10": results_dict["Recall@10"],
        "MAP@10": results_dict["MAP@10"],
        "NDCG@10": results_dict["NDCG@10"],
        "Coverage@10": results_dict["Coverage@10"]
    }


# =========================
# Table 1: Validation All
# =========================

validation_all_rows = [
    {
        "Model": "Popularity",
        **extract_at_10_metrics(popularity_results_validation_all)
    },
    {
        "Model": "ALS",
        **extract_at_10_metrics(best_als_results_validation_all)
    },
    {
        "Model": "BPR",
        **extract_at_10_metrics(best_bpr_results_validation_all)
    },
    {
        "Model": "Content-Based",
        **extract_at_10_metrics(best_content_results_validation_all)
    }
]

table_1_validation_all = pd.DataFrame(validation_all_rows)

table_1_validation_all = table_1_validation_all.sort_values(
    by="NDCG@10",
    ascending=False
).reset_index(drop=True)


# =========================
# Table 2: Validation High Intent
# =========================

validation_high_intent_rows = [
    {
        "Model": "Popularity",
        **extract_at_10_metrics(popularity_results_validation_high_intent)
    },
    {
        "Model": "ALS",
        **extract_at_10_metrics(best_als_results_validation_high_intent)
    },
    {
        "Model": "BPR",
        **extract_at_10_metrics(best_bpr_results_validation_high_intent)
    },
    {
        "Model": "Content-Based",
        **extract_at_10_metrics(best_content_high_intent_metrics)
    }
]

table_2_validation_high_intent = pd.DataFrame(validation_high_intent_rows)

table_2_validation_high_intent = table_2_validation_high_intent.sort_values(
    by="NDCG@10",
    ascending=False
).reset_index(drop=True)


# =========================
# Display tables
# =========================

print("TABLE 1 - VALIDATION ALL")
display(table_1_validation_all)

print("\nTABLE 2 - VALIDATION HIGH INTENT")
display(table_2_validation_high_intent)


# =========================
# Save tables
# =========================

table_1_path = REPORTS_DIR / "table_1_validation_all_model_comparison.csv"
table_2_path = REPORTS_DIR / "table_2_validation_high_intent_model_comparison.csv"

table_1_validation_all.to_csv(table_1_path, index=False)
table_2_validation_high_intent.to_csv(table_2_path, index=False)

print("Saved Table 1 to:")
print(table_1_path)

print("\nSaved Table 2 to:")
print(table_2_path)

TABLE 1 - VALIDATION ALL


,Model,Precision@10,Recall@10,MAP@10,NDCG@10,Coverage@10
0,ALS,0.030976,0.202025,0.083306,0.120246,0.487434
1,Content-Based,0.023154,0.158942,0.062760,0.091241,0.904357
2,BPR,0.019345,0.127791,0.050793,0.074674,0.794831
3,Popularity,0.001526,0.007657,0.002384,0.004200,0.001317



TABLE 2 - VALIDATION HIGH INTENT


,Model,Precision@10,Recall@10,MAP@10,NDCG@10,Coverage@10
0,ALS,0.025265,0.196525,0.084815,0.115658,0.298178
1,Content-Based,0.018142,0.145122,0.063107,0.085284,0.522662
2,BPR,0.013142,0.104546,0.043731,0.060335,0.460821
3,Popularity,0.003584,0.023781,0.011498,0.015406,0.001317


Saved Table 1 to:
C:\Users\pc\Documents\recommender_project\trial two\reports\table_1_validation_all_model_comparison.csv

Saved Table 2 to:
C:\Users\pc\Documents\recommender_project\trial two\reports\table_2_validation_high_intent_model_comparison.csv


In [59]:
# =========================
# Hybrid - Generate candidate recommendations
# =========================

HYBRID_CANDIDATE_K = 100

validation_users_hi = list(validation_gt_high_intent.keys())

# ALS candidates
als_candidates_validation_hi = generate_als_recommendations(
    model=best_als_model,
    user_ids=validation_users_hi,
    user_item_matrix=train_user_item_matrix,
    k=HYBRID_CANDIDATE_K
)

# Content-Based candidates
content_candidates_validation_hi = generate_content_based_recommendations_fast(
    users=validation_users_hi,
    user_content_profiles=best_user_content_profiles,
    item_content_matrix=best_item_content_matrix,
    user_seen_items_idx=user_seen_items_idx,
    k=HYBRID_CANDIDATE_K,
    batch_size=500
)

print("Hybrid candidate recommendations generated.")
print("Users:", len(validation_users_hi))

example_user = validation_users_hi[0]
print("\nExample user:", example_user)
print("ALS candidates:", als_candidates_validation_hi[example_user][:10])
print("Content candidates:", content_candidates_validation_hi[example_user][:10])

Processed users 500/2,260
Processed users 1,000/2,260
Processed users 1,500/2,260
Processed users 2,000/2,260
Processed users 2,260/2,260
Hybrid candidate recommendations generated.
Users: 2260

Example user: 5
ALS candidates: [6180, 7433, 10215, 5229, 4062, 15240, 16771, 14831, 13935, 15573]
Content candidates: [10215, 6675, 8944, 13665, 6916, 1171, 2412, 17248, 16769, 13807]


In [60]:
# =========================
# Hybrid - Rank fusion function
# =========================

def generate_hybrid_rank_fusion_recommendations(
    users,
    als_recommendations,
    content_recommendations,
    als_weight=0.8,
    content_weight=0.2,
    k=20
):
    """
    Combine ALS and Content-Based rankings using weighted rank fusion.
    """
    hybrid_recommendations = {}

    for user_idx in users:
        als_items = als_recommendations.get(user_idx, [])
        content_items = content_recommendations.get(user_idx, [])

        item_scores = {}

        # ALS rank scores
        for rank, item_idx in enumerate(als_items, start=1):
            item_scores[item_idx] = item_scores.get(item_idx, 0.0) + als_weight * (1.0 / rank)

        # Content-Based rank scores
        for rank, item_idx in enumerate(content_items, start=1):
            item_scores[item_idx] = item_scores.get(item_idx, 0.0) + content_weight * (1.0 / rank)

        # Sort by hybrid score
        ranked_items = sorted(
            item_scores.items(),
            key=lambda x: x[1],
            reverse=True
        )

        hybrid_recommendations[user_idx] = [
            item_idx for item_idx, score in ranked_items[:k]
        ]

    return hybrid_recommendations


print("Hybrid rank fusion function is ready.")

Hybrid rank fusion function is ready.


In [61]:
# =========================
# Hybrid - Tune weights on validation high-intent
# =========================

hybrid_weight_configs = [
    {"als_weight": 0.9, "content_weight": 0.1},
    {"als_weight": 0.8, "content_weight": 0.2},
    {"als_weight": 0.7, "content_weight": 0.3},
    {"als_weight": 0.6, "content_weight": 0.4},
    {"als_weight": 0.5, "content_weight": 0.5},
]

hybrid_tuning_results = []

NUM_ITEMS = train_user_item_matrix.shape[1]

for config in hybrid_weight_configs:
    print("\n" + "=" * 70)
    print("Hybrid config:", config)
    print("=" * 70)

    hybrid_recs_hi = generate_hybrid_rank_fusion_recommendations(
        users=validation_users_hi,
        als_recommendations=als_candidates_validation_hi,
        content_recommendations=content_candidates_validation_hi,
        als_weight=config["als_weight"],
        content_weight=config["content_weight"],
        k=20
    )

    hybrid_results_hi = evaluate_recommendations(
        recommendations=hybrid_recs_hi,
        ground_truth=validation_gt_high_intent,
        num_items=NUM_ITEMS,
        k_values=[5, 10, 20]
    )

    row = {
        "model": "Hybrid_ALS_Content",
        **config,
        **hybrid_results_hi
    }

    hybrid_tuning_results.append(row)

    print("Precision@10:", round(hybrid_results_hi["Precision@10"], 6))
    print("Recall@10:", round(hybrid_results_hi["Recall@10"], 6))
    print("MAP@10:", round(hybrid_results_hi["MAP@10"], 6))
    print("NDCG@10:", round(hybrid_results_hi["NDCG@10"], 6))
    print("Coverage@10:", round(hybrid_results_hi["Coverage@10"], 6))


hybrid_tuning_results_df = pd.DataFrame(hybrid_tuning_results)

hybrid_tuning_results_df = hybrid_tuning_results_df.sort_values(
    by="NDCG@10",
    ascending=False
).reset_index(drop=True)

display(hybrid_tuning_results_df)

hybrid_tuning_path = REPORTS_DIR / "hybrid_tuning_validation_high_intent.csv"
hybrid_tuning_results_df.to_csv(hybrid_tuning_path, index=False)

print("Saved Hybrid tuning results to:")
print(hybrid_tuning_path)


Hybrid config: {'als_weight': 0.9, 'content_weight': 0.1}
Precision@10: 0.026903
Recall@10: 0.209858
MAP@10: 0.086711
NDCG@10: 0.120238
Coverage@10: 0.345533

Hybrid config: {'als_weight': 0.8, 'content_weight': 0.2}
Precision@10: 0.027655
Recall@10: 0.217056
MAP@10: 0.088316
NDCG@10: 0.123198
Coverage@10: 0.38378

Hybrid config: {'als_weight': 0.7, 'content_weight': 0.3}
Precision@10: 0.027389
Recall@10: 0.218193
MAP@10: 0.089163
NDCG@10: 0.123924
Coverage@10: 0.416155

Hybrid config: {'als_weight': 0.6, 'content_weight': 0.4}
Precision@10: 0.027124
Recall@10: 0.216204
MAP@10: 0.088792
NDCG@10: 0.12313
Coverage@10: 0.446389

Hybrid config: {'als_weight': 0.5, 'content_weight': 0.5}
Precision@10: 0.026593
Recall@10: 0.212593
MAP@10: 0.088344
NDCG@10: 0.121735
Coverage@10: 0.465485


,model,als_weight,content_weight,Precision@5,Recall@5,MAP@5,NDCG@5,Coverage@5,Precision@10,Recall@10,MAP@10,NDCG@10,Coverage@10,Precision@20,Recall@20,MAP@20,NDCG@20,Coverage@20
0,Hybrid_ALS_Content,0.7,0.3,0.036903,0.147104,0.079128,0.100052,0.255048,0.027389,0.218193,0.089163,0.123924,0.416155,0.019115,0.297687,0.095085,0.145351,0.594765
1,Hybrid_ALS_Content,0.8,0.2,0.036549,0.145814,0.078336,0.099083,0.247366,0.027655,0.217056,0.088316,0.123198,0.383780,0.019159,0.294922,0.094211,0.144211,0.556409
2,Hybrid_ALS_Content,0.6,0.4,0.036018,0.143638,0.078622,0.098815,0.289289,0.027124,0.216204,0.088792,0.123130,0.446389,0.018872,0.295262,0.094824,0.144455,0.627634
3,Hybrid_ALS_Content,0.5,0.5,0.035398,0.141536,0.078585,0.098001,0.296971,0.026593,0.212593,0.088344,0.121735,0.465485,0.018761,0.294172,0.094475,0.143560,0.649473
4,Hybrid_ALS_Content,0.9,0.1,0.034690,0.137195,0.076657,0.095590,0.196499,0.026903,0.209858,0.086711,0.120238,0.345533,0.018894,0.290950,0.092743,0.141920,0.507188


Saved Hybrid tuning results to:
C:\Users\pc\Documents\recommender_project\trial two\reports\hybrid_tuning_validation_high_intent.csv


In [62]:
# =========================
# Save Best Hybrid high-intent result
# =========================

BEST_HYBRID_ALS_WEIGHT = 0.7
BEST_HYBRID_CONTENT_WEIGHT = 0.3

best_hybrid_high_intent_metrics = hybrid_tuning_results_df.iloc[0].to_dict()

best_hybrid_params = {
    "model": "Hybrid_ALS_Content",
    "method": "weighted_rank_fusion",
    "als_weight": BEST_HYBRID_ALS_WEIGHT,
    "content_weight": BEST_HYBRID_CONTENT_WEIGHT,
    "candidate_k": HYBRID_CANDIDATE_K
}

best_hybrid_params_path = MODELS_DIR / "best_hybrid_params.json"

with open(best_hybrid_params_path, "w") as f:
    json.dump(best_hybrid_params, f, indent=4)

print("Saved Best Hybrid params to:")
print(best_hybrid_params_path)

print("\nBest Hybrid High-Intent Metrics:")
for metric in ["Precision@10", "Recall@10", "MAP@10", "NDCG@10", "Coverage@10"]:
    print(f"{metric}: {best_hybrid_high_intent_metrics[metric]:.6f}")

Saved Best Hybrid params to:
C:\Users\pc\Documents\recommender_project\trial two\models\best_hybrid_params.json

Best Hybrid High-Intent Metrics:
Precision@10: 0.027389
Recall@10: 0.218193
MAP@10: 0.089163
NDCG@10: 0.123924
Coverage@10: 0.416155


In [63]:
# =========================
# Hybrid - Evaluate Best Hybrid on Validation All
# =========================

validation_users_all = list(validation_gt_all.keys())

# ALS candidates for validation all
als_candidates_validation_all = generate_als_recommendations(
    model=best_als_model,
    user_ids=validation_users_all,
    user_item_matrix=train_user_item_matrix,
    k=HYBRID_CANDIDATE_K
)

# Content-Based candidates for validation all
content_candidates_validation_all = generate_content_based_recommendations_fast(
    users=validation_users_all,
    user_content_profiles=best_user_content_profiles,
    item_content_matrix=best_item_content_matrix,
    user_seen_items_idx=user_seen_items_idx,
    k=HYBRID_CANDIDATE_K,
    batch_size=500
)

# Hybrid recommendations for validation all
best_hybrid_recs_validation_all = generate_hybrid_rank_fusion_recommendations(
    users=validation_users_all,
    als_recommendations=als_candidates_validation_all,
    content_recommendations=content_candidates_validation_all,
    als_weight=BEST_HYBRID_ALS_WEIGHT,
    content_weight=BEST_HYBRID_CONTENT_WEIGHT,
    k=20
)

best_hybrid_results_validation_all = evaluate_recommendations(
    recommendations=best_hybrid_recs_validation_all,
    ground_truth=validation_gt_all,
    num_items=train_user_item_matrix.shape[1],
    k_values=[5, 10, 20]
)

print_results(
    "BEST HYBRID - VALIDATION ALL",
    best_hybrid_results_validation_all
)

Processed users 500/22,869
Processed users 1,000/22,869
Processed users 1,500/22,869
Processed users 2,000/22,869
Processed users 2,500/22,869
Processed users 3,000/22,869
Processed users 3,500/22,869
Processed users 4,000/22,869
Processed users 4,500/22,869
Processed users 5,000/22,869
Processed users 5,500/22,869
Processed users 6,000/22,869
Processed users 6,500/22,869
Processed users 7,000/22,869
Processed users 7,500/22,869
Processed users 8,000/22,869
Processed users 8,500/22,869
Processed users 9,000/22,869
Processed users 9,500/22,869
Processed users 10,000/22,869
Processed users 10,500/22,869
Processed users 11,000/22,869
Processed users 11,500/22,869
Processed users 12,000/22,869
Processed users 12,500/22,869
Processed users 13,000/22,869
Processed users 13,500/22,869
Processed users 14,000/22,869
Processed users 14,500/22,869
Processed users 15,000/22,869
Processed users 15,500/22,869
Processed users 16,000/22,869
Processed users 16,500/22,869
Processed users 17,000/22,869
P

In [64]:
# =========================
# Save Best Hybrid validation results
# =========================

# High-intent metrics from best row in tuning
best_hybrid_high_intent_metrics = {
    "Precision@5": hybrid_tuning_results_df.iloc[0]["Precision@5"],
    "Recall@5": hybrid_tuning_results_df.iloc[0]["Recall@5"],
    "MAP@5": hybrid_tuning_results_df.iloc[0]["MAP@5"],
    "NDCG@5": hybrid_tuning_results_df.iloc[0]["NDCG@5"],
    "Coverage@5": hybrid_tuning_results_df.iloc[0]["Coverage@5"],

    "Precision@10": hybrid_tuning_results_df.iloc[0]["Precision@10"],
    "Recall@10": hybrid_tuning_results_df.iloc[0]["Recall@10"],
    "MAP@10": hybrid_tuning_results_df.iloc[0]["MAP@10"],
    "NDCG@10": hybrid_tuning_results_df.iloc[0]["NDCG@10"],
    "Coverage@10": hybrid_tuning_results_df.iloc[0]["Coverage@10"],

    "Precision@20": hybrid_tuning_results_df.iloc[0]["Precision@20"],
    "Recall@20": hybrid_tuning_results_df.iloc[0]["Recall@20"],
    "MAP@20": hybrid_tuning_results_df.iloc[0]["MAP@20"],
    "NDCG@20": hybrid_tuning_results_df.iloc[0]["NDCG@20"],
    "Coverage@20": hybrid_tuning_results_df.iloc[0]["Coverage@20"],
}

best_hybrid_validation_all_row = {
    "model": "Best_Hybrid_ALS_Content",
    "evaluation_type": "validation_all",
    "als_weight": BEST_HYBRID_ALS_WEIGHT,
    "content_weight": BEST_HYBRID_CONTENT_WEIGHT,
    "candidate_k": HYBRID_CANDIDATE_K,
    **best_hybrid_results_validation_all
}

best_hybrid_validation_high_intent_row = {
    "model": "Best_Hybrid_ALS_Content",
    "evaluation_type": "validation_high_intent",
    "als_weight": BEST_HYBRID_ALS_WEIGHT,
    "content_weight": BEST_HYBRID_CONTENT_WEIGHT,
    "candidate_k": HYBRID_CANDIDATE_K,
    **best_hybrid_high_intent_metrics
}

best_hybrid_results_df = pd.DataFrame([
    best_hybrid_validation_all_row,
    best_hybrid_validation_high_intent_row
])

best_hybrid_results_path = REPORTS_DIR / "best_hybrid_validation_results.csv"
best_hybrid_results_df.to_csv(best_hybrid_results_path, index=False)

print("Saved Best Hybrid validation results to:")
print(best_hybrid_results_path)

display(best_hybrid_results_df)

Saved Best Hybrid validation results to:
C:\Users\pc\Documents\recommender_project\trial two\reports\best_hybrid_validation_results.csv


,model,evaluation_type,als_weight,content_weight,candidate_k,Precision@5,Recall@5,MAP@5,NDCG@5,Coverage@5,Precision@10,Recall@10,MAP@10,NDCG@10,Coverage@10,Precision@20,Recall@20,MAP@20,NDCG@20,Coverage@20
0,Best_Hybrid_ALS_Content,validation_all,0.7,0.3,100,0.041077,0.138201,0.075020,0.098318,0.651778,0.032516,0.216869,0.086595,0.126230,0.823529,0.024758,0.323292,0.095246,0.156278,0.916978
1,Best_Hybrid_ALS_Content,validation_high_intent,0.7,0.3,100,0.036903,0.147104,0.079128,0.100052,0.255048,0.027389,0.218193,0.089163,0.123924,0.416155,0.019115,0.297687,0.095085,0.145351,0.594765


In [65]:
# =========================
# Final model comparison tables with Hybrid
# =========================

def extract_at_10_metrics(results_dict):
    """
    Extract @10 metrics from a model results dictionary.
    """
    return {
        "Precision@10": results_dict["Precision@10"],
        "Recall@10": results_dict["Recall@10"],
        "MAP@10": results_dict["MAP@10"],
        "NDCG@10": results_dict["NDCG@10"],
        "Coverage@10": results_dict["Coverage@10"]
    }


# Table 1: Validation All
final_validation_all_rows = [
    {
        "Model": "Popularity",
        **extract_at_10_metrics(popularity_results_validation_all)
    },
    {
        "Model": "ALS",
        **extract_at_10_metrics(best_als_results_validation_all)
    },
    {
        "Model": "BPR",
        **extract_at_10_metrics(best_bpr_results_validation_all)
    },
    {
        "Model": "Content-Based",
        **extract_at_10_metrics(best_content_results_validation_all)
    },
    {
        "Model": "Hybrid ALS + Content",
        **extract_at_10_metrics(best_hybrid_results_validation_all)
    }
]

final_table_1_validation_all = pd.DataFrame(final_validation_all_rows)

final_table_1_validation_all = final_table_1_validation_all.sort_values(
    by="NDCG@10",
    ascending=False
).reset_index(drop=True)


# Table 2: Validation High Intent
final_validation_high_intent_rows = [
    {
        "Model": "Popularity",
        **extract_at_10_metrics(popularity_results_validation_high_intent)
    },
    {
        "Model": "ALS",
        **extract_at_10_metrics(best_als_results_validation_high_intent)
    },
    {
        "Model": "BPR",
        **extract_at_10_metrics(best_bpr_results_validation_high_intent)
    },
    {
        "Model": "Content-Based",
        **extract_at_10_metrics(best_content_high_intent_metrics)
    },
    {
        "Model": "Hybrid ALS + Content",
        **extract_at_10_metrics(best_hybrid_high_intent_metrics)
    }
]

final_table_2_validation_high_intent = pd.DataFrame(final_validation_high_intent_rows)

final_table_2_validation_high_intent = final_table_2_validation_high_intent.sort_values(
    by="NDCG@10",
    ascending=False
).reset_index(drop=True)


# Display rounded tables
metric_cols = ["Precision@10", "Recall@10", "MAP@10", "NDCG@10", "Coverage@10"]

final_table_1_display = final_table_1_validation_all.copy()
final_table_2_display = final_table_2_validation_high_intent.copy()

final_table_1_display[metric_cols] = final_table_1_display[metric_cols].round(6)
final_table_2_display[metric_cols] = final_table_2_display[metric_cols].round(6)

print("FINAL TABLE 1 - VALIDATION ALL")
display(final_table_1_display)

print("\nFINAL TABLE 2 - VALIDATION HIGH INTENT")
display(final_table_2_display)

# Save final tables
final_table_1_path = REPORTS_DIR / "final_table_1_validation_all_model_comparison.csv"
final_table_2_path = REPORTS_DIR / "final_table_2_validation_high_intent_model_comparison.csv"

final_table_1_validation_all.to_csv(final_table_1_path, index=False)
final_table_2_validation_high_intent.to_csv(final_table_2_path, index=False)

print("Saved final Table 1 to:")
print(final_table_1_path)

print("\nSaved final Table 2 to:")
print(final_table_2_path)

FINAL TABLE 1 - VALIDATION ALL


,Model,Precision@10,Recall@10,MAP@10,NDCG@10,Coverage@10
0,Hybrid ALS + Content,0.032516,0.216869,0.086595,0.126230,0.823529
1,ALS,0.030976,0.202025,0.083306,0.120246,0.487434
2,Content-Based,0.023154,0.158942,0.062760,0.091241,0.904357
3,BPR,0.019345,0.127791,0.050793,0.074674,0.794831
4,Popularity,0.001526,0.007657,0.002384,0.004200,0.001317



FINAL TABLE 2 - VALIDATION HIGH INTENT


,Model,Precision@10,Recall@10,MAP@10,NDCG@10,Coverage@10
0,Hybrid ALS + Content,0.027389,0.218193,0.089163,0.123924,0.416155
1,ALS,0.025265,0.196525,0.084815,0.115658,0.298178
2,Content-Based,0.018142,0.145122,0.063107,0.085284,0.522662
3,BPR,0.013142,0.104546,0.043731,0.060335,0.460821
4,Popularity,0.003584,0.023781,0.011498,0.015406,0.001317


Saved final Table 1 to:
C:\Users\pc\Documents\recommender_project\trial two\reports\final_table_1_validation_all_model_comparison.csv

Saved final Table 2 to:
C:\Users\pc\Documents\recommender_project\trial two\reports\final_table_2_validation_high_intent_model_comparison.csv


## Final Validation Comparison

The Hybrid ALS + Content-Based model achieved the best validation performance across both evaluation settings.

On **Validation High Intent**, the Hybrid model achieved the highest NDCG@10 and Recall@10:

- NDCG@10 = 0.123924
- Recall@10 = 0.218193
- Coverage@10 = 0.416155

Compared with standalone ALS, the Hybrid model improved both ranking quality and coverage. This shows that combining ALS collaborative filtering with content-based item metadata helps produce more accurate and more diverse recommendations.

Therefore, the Hybrid ALS + Content-Based model is selected as the best validation candidate. Final confirmation should be done later using the held-out test set only once, after all model selection is complete.

In [66]:
# =========================
# Hybrid - Regenerate Best Hybrid recommendations for validation high-intent
# =========================

best_hybrid_recs_validation_high_intent = generate_hybrid_rank_fusion_recommendations(
    users=validation_users_hi,
    als_recommendations=als_candidates_validation_hi,
    content_recommendations=content_candidates_validation_hi,
    als_weight=BEST_HYBRID_ALS_WEIGHT,
    content_weight=BEST_HYBRID_CONTENT_WEIGHT,
    k=20
)

print("Best Hybrid validation high-intent recommendations regenerated.")
print("Users:", len(best_hybrid_recs_validation_high_intent))

Best Hybrid validation high-intent recommendations regenerated.
Users: 2260


In [67]:
# =========================
# Hybrid - Sanity check
# =========================

hybrid_recommendation_issues = check_recommendation_sanity(
    recommendations=best_hybrid_recs_validation_high_intent,
    user_seen_items_idx=user_seen_items_idx,
    num_items=train_user_item_matrix.shape[1],
    name="Best Hybrid - Validation High Intent"
)

CHECK 2: RECOMMENDATION SANITY - Best Hybrid - Validation High Intent
Users checked: 2260
Users with already-seen item leakage: 0
Users with invalid item ids: 0
Users with duplicate recommendations: 0
Users with empty recommendations: 0

Recommendation length summary:
count    2260.0
mean       20.0
std         0.0
min        20.0
25%        20.0
50%        20.0
75%        20.0
max        20.0
dtype: float64

------------------------------------------------------------
PASS: Recommendation outputs look valid.


In [68]:
# =========================
# FINAL TEST - Hybrid evaluation on test high-intent
# =========================

test_users_hi = list(test_gt_high_intent.keys())

# ALS candidates for test high-intent
als_candidates_test_hi = generate_als_recommendations(
    model=best_als_model,
    user_ids=test_users_hi,
    user_item_matrix=train_user_item_matrix,
    k=HYBRID_CANDIDATE_K
)

# Content-Based candidates for test high-intent
content_candidates_test_hi = generate_content_based_recommendations_fast(
    users=test_users_hi,
    user_content_profiles=best_user_content_profiles,
    item_content_matrix=best_item_content_matrix,
    user_seen_items_idx=user_seen_items_idx,
    k=HYBRID_CANDIDATE_K,
    batch_size=500
)

# Hybrid recommendations for test high-intent
best_hybrid_recs_test_high_intent = generate_hybrid_rank_fusion_recommendations(
    users=test_users_hi,
    als_recommendations=als_candidates_test_hi,
    content_recommendations=content_candidates_test_hi,
    als_weight=BEST_HYBRID_ALS_WEIGHT,
    content_weight=BEST_HYBRID_CONTENT_WEIGHT,
    k=20
)

best_hybrid_results_test_high_intent = evaluate_recommendations(
    recommendations=best_hybrid_recs_test_high_intent,
    ground_truth=test_gt_high_intent,
    num_items=train_user_item_matrix.shape[1],
    k_values=[5, 10, 20]
)

print_results(
    "FINAL HYBRID - TEST HIGH INTENT",
    best_hybrid_results_test_high_intent
)

Processed users 500/3,178
Processed users 1,000/3,178
Processed users 1,500/3,178
Processed users 2,000/3,178
Processed users 2,500/3,178
Processed users 3,000/3,178
Processed users 3,178/3,178

FINAL HYBRID - TEST HIGH INTENT
Precision@5: 0.037948
Recall@5: 0.154393
MAP@5: 0.087637
NDCG@5: 0.108341
Coverage@5: 0.300044
Precision@10: 0.027533
Recall@10: 0.219314
MAP@10: 0.096398
NDCG@10: 0.130149
Coverage@10: 0.474704
Precision@20: 0.019478
Recall@20: 0.304470
MAP@20: 0.102867
NDCG@20: 0.153001
Coverage@20: 0.657814


In [69]:
# =========================
# Compare validation vs test high-intent
# =========================

validation_high_intent_summary = {
    "Precision@10": best_hybrid_high_intent_metrics["Precision@10"],
    "Recall@10": best_hybrid_high_intent_metrics["Recall@10"],
    "MAP@10": best_hybrid_high_intent_metrics["MAP@10"],
    "NDCG@10": best_hybrid_high_intent_metrics["NDCG@10"],
    "Coverage@10": best_hybrid_high_intent_metrics["Coverage@10"]
}

test_high_intent_summary = {
    "Precision@10": best_hybrid_results_test_high_intent["Precision@10"],
    "Recall@10": best_hybrid_results_test_high_intent["Recall@10"],
    "MAP@10": best_hybrid_results_test_high_intent["MAP@10"],
    "NDCG@10": best_hybrid_results_test_high_intent["NDCG@10"],
    "Coverage@10": best_hybrid_results_test_high_intent["Coverage@10"]
}

validation_test_comparison = pd.DataFrame([
    {
        "split": "validation_high_intent",
        **validation_high_intent_summary
    },
    {
        "split": "test_high_intent",
        **test_high_intent_summary
    }
])

display(validation_test_comparison)

# Drop percentage for NDCG@10
validation_ndcg = validation_high_intent_summary["NDCG@10"]
test_ndcg = test_high_intent_summary["NDCG@10"]

ndcg_drop_percentage = (
    (validation_ndcg - test_ndcg) / validation_ndcg
) * 100

print(f"Validation NDCG@10: {validation_ndcg:.6f}")
print(f"Test NDCG@10: {test_ndcg:.6f}")
print(f"NDCG@10 drop percentage: {ndcg_drop_percentage:.2f}%")

,split,Precision@10,Recall@10,MAP@10,NDCG@10,Coverage@10
0,validation_high_intent,0.027389,0.218193,0.089163,0.123924,0.416155
1,test_high_intent,0.027533,0.219314,0.096398,0.130149,0.474704


Validation NDCG@10: 0.123924
Test NDCG@10: 0.130149
NDCG@10 drop percentage: -5.02%


In [70]:
#No clear evidence of overfitting.

In [ ]:
## Final Test Evaluation

After selecting the Hybrid ALS + Content-Based model using validation results, the final model was evaluated once on the held-out high-intent test set.

The final Hybrid model achieved:

- Precision@10 = 0.027533
- Recall@10 = 0.219314
- MAP@10 = 0.096398
- NDCG@10 = 0.130149
- Coverage@10 = 0.474704

Compared with validation performance, the test NDCG@10 increased from 0.123924 to 0.130149. This corresponds to a -5.02% drop, meaning the model performed slightly better on the test set than on validation.

This indicates that there is no clear evidence of overfitting. The model generalizes well to unseen high-intent user interactions.

In [71]:
# =========================
# Save final Hybrid test results
# =========================

final_hybrid_test_results_row = {
    "model": "Final_Hybrid_ALS_Content",
    "evaluation_type": "test_high_intent",
    "als_weight": BEST_HYBRID_ALS_WEIGHT,
    "content_weight": BEST_HYBRID_CONTENT_WEIGHT,
    "candidate_k": HYBRID_CANDIDATE_K,
    **best_hybrid_results_test_high_intent
}

final_hybrid_test_results_df = pd.DataFrame([final_hybrid_test_results_row])

final_test_results_path = REPORTS_DIR / "final_hybrid_test_high_intent_results.csv"
final_hybrid_test_results_df.to_csv(final_test_results_path, index=False)

validation_test_comparison_path = REPORTS_DIR / "hybrid_validation_vs_test_high_intent.csv"
validation_test_comparison.to_csv(validation_test_comparison_path, index=False)

print("Saved final Hybrid test results to:")
print(final_test_results_path)

print("\nSaved validation vs test comparison to:")
print(validation_test_comparison_path)

display(final_hybrid_test_results_df)

Saved final Hybrid test results to:
C:\Users\pc\Documents\recommender_project\trial two\reports\final_hybrid_test_high_intent_results.csv

Saved validation vs test comparison to:
C:\Users\pc\Documents\recommender_project\trial two\reports\hybrid_validation_vs_test_high_intent.csv


,model,evaluation_type,als_weight,content_weight,candidate_k,Precision@5,Recall@5,MAP@5,NDCG@5,Coverage@5,Precision@10,Recall@10,MAP@10,NDCG@10,Coverage@10,Precision@20,Recall@20,MAP@20,NDCG@20,Coverage@20
0,Final_Hybrid_ALS_Content,test_high_intent,0.7,0.3,100,0.037948,0.154393,0.087637,0.108341,0.300044,0.027533,0.219314,0.096398,0.130149,0.474704,0.019478,0.30447,0.102867,0.153001,0.657814


In [ ]:
#Final selected model = Hybrid ALS + Content-Based

In [72]:
# =========================
# Save final model selection summary
# =========================

final_model_summary = {
    "final_selected_model": "Hybrid ALS + Content-Based",
    "selection_reason": (
        "The hybrid model achieved the best validation ranking performance "
        "and improved coverage compared with standalone ALS."
    ),
    "hybrid_method": "Weighted Rank Fusion",
    "hybrid_parameters": {
        "als_weight": BEST_HYBRID_ALS_WEIGHT,
        "content_weight": BEST_HYBRID_CONTENT_WEIGHT,
        "candidate_k": HYBRID_CANDIDATE_K
    },
    "final_test_high_intent_results": {
        "Precision@10": best_hybrid_results_test_high_intent["Precision@10"],
        "Recall@10": best_hybrid_results_test_high_intent["Recall@10"],
        "MAP@10": best_hybrid_results_test_high_intent["MAP@10"],
        "NDCG@10": best_hybrid_results_test_high_intent["NDCG@10"],
        "Coverage@10": best_hybrid_results_test_high_intent["Coverage@10"]
    },
    "overfitting_check": {
        "validation_high_intent_NDCG@10": validation_high_intent_summary["NDCG@10"],
        "test_high_intent_NDCG@10": test_high_intent_summary["NDCG@10"],
        "ndcg_drop_percentage": ndcg_drop_percentage,
        "conclusion": "No clear evidence of overfitting."
    },
    "notes": [
        "ALS achieved the best standalone ranking quality.",
        "Content-Based filtering achieved strong coverage.",
        "The Hybrid model combined ALS accuracy with Content-Based coverage.",
        "Final evaluation was performed once on the held-out test set."
    ]
}

final_model_summary_path = REPORTS_DIR / "final_model_selection_summary.json"

with open(final_model_summary_path, "w") as f:
    json.dump(final_model_summary, f, indent=4)

print("Saved final model selection summary to:")
print(final_model_summary_path)

final_model_summary

Saved final model selection summary to:
C:\Users\pc\Documents\recommender_project\trial two\reports\final_model_selection_summary.json


{'final_selected_model': 'Hybrid ALS + Content-Based',
 'selection_reason': 'The hybrid model achieved the best validation ranking performance and improved coverage compared with standalone ALS.',
 'hybrid_method': 'Weighted Rank Fusion',
 'hybrid_parameters': {'als_weight': 0.7,
  'content_weight': 0.3,
  'candidate_k': 100},
 'final_test_high_intent_results': {'Precision@10': np.float64(0.02753303964757709),
  'Recall@10': np.float64(0.21931387595962978),
  'MAP@10': np.float64(0.09639828553091048),
  'NDCG@10': np.float64(0.1301487277672154),
  'Coverage@10': 0.4747036874451273},
 'overfitting_check': {'validation_high_intent_NDCG@10': np.float64(0.1239244059477315),
  'test_high_intent_NDCG@10': np.float64(0.1301487277672154),
  'ndcg_drop_percentage': np.float64(-5.022676341986393),
  'conclusion': 'No clear evidence of overfitting.'},
 'notes': ['ALS achieved the best standalone ranking quality.',
  'Content-Based filtering achieved strong coverage.',
  'The Hybrid model combined

In [73]:
# =========================
# Milestone 2 artifact check
# =========================

milestone_2_files = [
    REPORTS_DIR / "popularity_baseline_validation_results.csv",
    REPORTS_DIR / "best_als_validation_results.csv",
    REPORTS_DIR / "best_bpr_validation_results.csv",
    REPORTS_DIR / "best_content_based_validation_results.csv",
    REPORTS_DIR / "best_hybrid_validation_results.csv",
    REPORTS_DIR / "final_hybrid_test_high_intent_results.csv",
    REPORTS_DIR / "hybrid_validation_vs_test_high_intent.csv",
    REPORTS_DIR / "final_model_selection_summary.json",
    
    MODELS_DIR / "best_als_model.pkl",
    MODELS_DIR / "best_als_params.json",
    MODELS_DIR / "best_bpr_model.pkl",
    MODELS_DIR / "best_bpr_params.json",
    MODELS_DIR / "best_content_based_params.json",
    MODELS_DIR / "best_hybrid_params.json"
]

print("===== MILESTONE 2 ARTIFACT CHECK =====")

all_found = True

for file_path in milestone_2_files:
    if file_path.exists():
        print("FOUND:", file_path.name)
    else:
        print("MISSING:", file_path.name)
        all_found = False

if all_found:
    print("\nMilestone 2 core artifacts are complete.")
else:
    print("\nSome Milestone 2 files are missing.")

===== MILESTONE 2 ARTIFACT CHECK =====
FOUND: popularity_baseline_validation_results.csv
FOUND: best_als_validation_results.csv
FOUND: best_bpr_validation_results.csv
FOUND: best_content_based_validation_results.csv
FOUND: best_hybrid_validation_results.csv
FOUND: final_hybrid_test_high_intent_results.csv
FOUND: hybrid_validation_vs_test_high_intent.csv
FOUND: final_model_selection_summary.json
FOUND: best_als_model.pkl
FOUND: best_als_params.json
FOUND: best_bpr_model.pkl
FOUND: best_bpr_params.json
FOUND: best_content_based_params.json
FOUND: best_hybrid_params.json

Milestone 2 core artifacts are complete.
